## Overview

This repository documents an academic deep-learning project on **convolutional neural networks for the iCoSimal V3 dataset**. The goal of the work was to systematically explore how architecture design, hyperparameter choices, regularization, optimization, and transfer learning affect model performance and generalization.

The project covers several core experimental objectives, including:
- progressively increasing CNN complexity to study the effect of depth,
- analysing the importance of hyperparameter tuning,
- demonstrating underfitting, good fit, and overfitting behaviour,
- evaluating regularization techniques,
- comparing different optimization algorithms.

In addition, selected advanced topics were explored, including experiment tracking with Weights & Biases, automated hyperparameter search with Optuna, and transfer learning with pretrained architectures.

The repository is intended to document the experimental workflow, key results, and main lessons learned from the study.

## 1. Initial CNN Model Zoo for Depth Comparison

To investigate the benefit of depth, I first defined a small CNN model zoo with four architectures of increasing complexity: `ModelA`, `ModelB`, `ModelC`, and `ModelD`. The models share the same overall classification objective and differ mainly in the number of convolutional blocks and feature channels.

The purpose of this design is to enable a fair and interpretable comparison between shallow and deeper CNNs. Instead of changing many design choices at once, the architectures are progressively extended so that performance differences can be related mainly to model depth and representational capacity.

A global-average-pooling-based classification head is used in all models. This avoids hardcoding flatten dimensions and keeps the models flexible with respect to the input image resolution. This model zoo serves as the starting point for **Objective (a): showing the benefit of depth**. 


In [ ]:
# CNN model zoo for fair depth comparison
# Objective a): show the benefit of depth
# ============================================

from __future__ import annotations

import torch
import torch.nn as nn
from typing import Dict


# --------------------------------------------
# Small reusable building blocks
# --------------------------------------------
class ConvBlock(nn.Module):
    """
    One convolution block:
    Conv2d -> ReLU -> MaxPool2d
    """
    def __init__(self, in_channels: int, out_channels: int):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2)
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.block(x)


class DoubleConvBlock(nn.Module):
    """
    Deeper convolution block:
    Conv2d -> ReLU -> Conv2d -> ReLU -> MaxPool2d
    """
    def __init__(self, in_channels: int, out_channels: int):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2)
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.block(x)


class ClassificationHead(nn.Module):
    """
    Global average pooling based classification head.
    This avoids hardcoding flatten dimensions and keeps
    models flexible for different input image sizes.
    """
    def __init__(self, in_channels: int, hidden_dim: int, num_classes: int):
        super().__init__()
        self.head = nn.Sequential(
            nn.AdaptiveAvgPool2d((1, 1)),  # -> [B, C, 1, 1]
            nn.Flatten(),                  # -> [B, C]
            nn.Linear(in_channels, hidden_dim),
            nn.ReLU(inplace=True),
            nn.Linear(hidden_dim, num_classes)
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.head(x)


# --------------------------------------------
# Model A: very simple baseline
# 1 Conv-Block
# --------------------------------------------
class ModelA(nn.Module):
    """
    ModelA:
    Conv(16) -> ReLU -> Pool
    GAP -> Dense(64) -> ReLU -> Dense(num_classes)
    """
    def __init__(self, in_channels: int = 3, num_classes: int = 10):
        super().__init__()
        self.features = nn.Sequential(
            ConvBlock(in_channels, 16)
        )
        self.classifier = ClassificationHead(
            in_channels=16,
            hidden_dim=64,
            num_classes=num_classes
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.features(x)
        x = self.classifier(x)
        return x


# --------------------------------------------
# Model B: slightly deeper
# 2 Conv-Blocks
# --------------------------------------------
class ModelB(nn.Module):
    """
    ModelB:
    Conv(16) -> ReLU -> Pool
    Conv(32) -> ReLU -> Pool
    GAP -> Dense(128) -> ReLU -> Dense(num_classes)
    """
    def __init__(self, in_channels: int = 3, num_classes: int = 10):
        super().__init__()
        self.features = nn.Sequential(
            ConvBlock(in_channels, 16),
            ConvBlock(16, 32)
        )
        self.classifier = ClassificationHead(
            in_channels=32,
            hidden_dim=128,
            num_classes=num_classes
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.features(x)
        x = self.classifier(x)
        return x


# --------------------------------------------
# Model C: deeper / more expressive
# 2 blocks with 2 conv layers each
# --------------------------------------------
class ModelC(nn.Module):
    """
    ModelC:
    Conv(32) -> ReLU -> Conv(32) -> ReLU -> Pool
    Conv(64) -> ReLU -> Conv(64) -> ReLU -> Pool
    GAP -> Dense(128) -> ReLU -> Dense(num_classes)
    """
    def __init__(self, in_channels: int = 3, num_classes: int = 10):
        super().__init__()
        self.features = nn.Sequential(
            DoubleConvBlock(in_channels, 32),
            DoubleConvBlock(32, 64)
        )
        self.classifier = ClassificationHead(
            in_channels=64,
            hidden_dim=128,
            num_classes=num_classes
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.features(x)
        x = self.classifier(x)
        return x


# --------------------------------------------
# Model D: deepest model in this comparison
# 3 blocks with 2 conv layers each
# --------------------------------------------
class ModelD(nn.Module):
    """
    ModelD:
    Conv(32)  -> ReLU -> Conv(32)  -> ReLU -> Pool
    Conv(64)  -> ReLU -> Conv(64)  -> ReLU -> Pool
    Conv(128) -> ReLU -> Conv(128) -> ReLU -> Pool
    GAP -> Dense(128) -> ReLU -> Dense(num_classes)
    """
    def __init__(self, in_channels: int = 3, num_classes: int = 10):
        super().__init__()
        self.features = nn.Sequential(
            DoubleConvBlock(in_channels, 32),
            DoubleConvBlock(32, 64),
            DoubleConvBlock(64, 128)
        )
        self.classifier = ClassificationHead(
            in_channels=128,
            hidden_dim=128,
            num_classes=num_classes
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.features(x)
        x = self.classifier(x)
        return x


# --------------------------------------------
# Factory function
# --------------------------------------------
def create_model_zoo(
    num_classes: int = 10,
    in_channels: int = 3
) -> Dict[str, nn.Module]:
    """
    Returns all four models in a dictionary.
    """
    return {
        "ModelA": ModelA(in_channels=in_channels, num_classes=num_classes),
        "ModelB": ModelB(in_channels=in_channels, num_classes=num_classes),
        "ModelC": ModelC(in_channels=in_channels, num_classes=num_classes),
        "ModelD": ModelD(in_channels=in_channels, num_classes=num_classes),
    }


# --------------------------------------------
# Parameter counting helper
# --------------------------------------------
def count_trainable_parameters(model: nn.Module) -> int:
    """
    Count trainable parameters of a model.
    """
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


# --------------------------------------------
# Print a compact overview
# --------------------------------------------
def print_model_overview(
    models: Dict[str, nn.Module],
    input_shape: tuple[int, int, int, int] = (1, 3, 64, 64),
    device: str | torch.device = "cpu"
) -> None:
    """
    Prints parameter counts and output shapes for a dummy batch.
    """
    x = torch.randn(*input_shape).to(device)

    for name, model in models.items():
        model = model.to(device)
        model.eval()
        with torch.no_grad():
            y = model(x)
        n_params = count_trainable_parameters(model)
        print(f"{name}:")
        print(f"  trainable params: {n_params:,}")
        print(f"  output shape    : {tuple(y.shape)}")
        print("-" * 40)


# --------------------------------------------
# Example usage in a notebook cell
# --------------------------------------------
if __name__ == "__main__":
    # Device selection 
    if torch.backends.mps.is_available():
        device = torch.device("mps")
    elif torch.mps.is_available():
        device = torch.device("mps")
    else:
        device = torch.device("cpu")

    models = create_model_zoo(num_classes=10, in_channels=3)
    print_model_overview(models, input_shape=(1, 3, 64, 64), device=device)


### Initial observations

The initial model overview confirms that all four architectures produce the expected output shape for a 10-class classification problem. As intended, the number of trainable parameters increases from `ModelA` to `ModelD`, reflecting the increasing depth and representational capacity of the models.

At this stage, these results only verify that the architectures are implemented correctly and differ in a controlled way. Whether increased depth actually improves performance will be evaluated later based on training and validation loss, accuracy, and generalisation behaviour.

## 2. Objective (a): Fair Training and Comparison of ModelA-ModelD

To investigate the benefit of depth in a controlled way, I trained the four CNN architectures (`ModelA` to `ModelD`) under the same initial training setup. This experiment addresses **Objective (a)** of the assignment by comparing progressively deeper and more expressive CNNs under identical conditions. 

For this comparison, all models were trained with the same dataset split, optimizer type, learning rate, batch size, image resolution, and loss function. In addition, no data augmentation and no weight decay were used at this stage. This keeps the setup intentionally simple and makes the comparison more interpretable, since performance differences can be attributed mainly to architectural depth and model capacity rather than to additional training tricks.

To keep the first comparison computationally manageable, all images were resized to **64x64** and each model was trained for **10 epochs**. This is long enough to observe meaningful learning behaviour while still keeping the experiment suitable as a fair baseline comparison rather than as a fully optimised final benchmark.

In [ ]:
# FAIR TRAINING + COMPARISON PIPELINE FOR ModelA...ModelD
# =========================================================

from __future__ import annotations

import copy
import random
from pathlib import Path
from dataclasses import dataclass

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms


# ---------------------------------------------------------
# 1) Configuration
# ---------------------------------------------------------
@dataclass
class Config:
    image_size: int = 64           
    batch_size: int = 64
    num_workers: int = 0           # safer in notebooks
    lr: float = 1e-3
    weight_decay: float = 0.0      
    max_epochs: int = 10
    early_stopping_patience: int = 4
    use_early_stopping: bool = True
    seed: int = 42
    save_best_weights: bool = True
    save_dir: str = "./saved_models_depth_study"
    in_channels: int = 3           # RGB images
    num_classes: int = 10          # iCoSimal V3: 10 animal classes


CFG = Config()


# ---------------------------------------------------------
# 2) Reproducibility
# ---------------------------------------------------------
def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    # deterministic flags (may reduce speed a bit, but improves reproducibility)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


set_seed(CFG.seed)


# ---------------------------------------------------------
# 3) Device selection
# ---------------------------------------------------------
def get_device() -> torch.device:
    if torch.backends.mps.is_available():
        return torch.device("mps")
    if torch.cuda.is_available():
        return torch.device("cuda")
    return torch.device("cpu")


DEVICE = get_device()
print("Using device:", DEVICE)


# ---------------------------------------------------------
# 4) Paths
# ---------------------------------------------------------
TRAIN_DIR = "data/train"
VAL_DIR = "data/validate"

# ---------------------------------------------------------
# 5) Transforms
# No augmentation yet -> fair comparison for objective a)
# ---------------------------------------------------------
train_transform = transforms.Compose([
    transforms.Resize((CFG.image_size, CFG.image_size)),
    transforms.ToTensor(),
])

val_transform = transforms.Compose([
    transforms.Resize((CFG.image_size, CFG.image_size)),
    transforms.ToTensor(),
])


# ---------------------------------------------------------
# 6) Datasets + DataLoaders
# ---------------------------------------------------------
train_dataset = datasets.ImageFolder(TRAIN_DIR, transform=train_transform)
val_dataset = datasets.ImageFolder(VAL_DIR, transform=val_transform)

class_names = train_dataset.classes
class_to_idx = train_dataset.class_to_idx

print("Classes:", class_names)
print("Class mapping:", class_to_idx)
print(f"Train samples: {len(train_dataset):,}")
print(f"Val samples  : {len(val_dataset):,}")

assert len(class_names) == CFG.num_classes, (
    f"Expected {CFG.num_classes} classes, got {len(class_names)}. "
    "Adjust CFG.num_classes if needed."
)

train_loader = DataLoader(
    train_dataset,
    batch_size=CFG.batch_size,
    shuffle=True,
    num_workers=CFG.num_workers,
    pin_memory=(DEVICE.type == "cuda"),
)

val_loader = DataLoader(
    val_dataset,
    batch_size=CFG.batch_size,
    shuffle=False,
    num_workers=CFG.num_workers,
    pin_memory=(DEVICE.type == "cuda"),
)


# ---------------------------------------------------------
# 7) Utilities
# ---------------------------------------------------------
def accuracy_from_logits(logits: torch.Tensor, targets: torch.Tensor) -> float:
    preds = logits.argmax(dim=1)
    correct = (preds == targets).sum().item()
    total = targets.size(0)
    return correct / total


def count_trainable_parameters(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


# ---------------------------------------------------------
# 8) One epoch train / validate
# ---------------------------------------------------------
def train_one_epoch(
    model: nn.Module,
    loader: DataLoader,
    criterion: nn.Module,
    optimizer: torch.optim.Optimizer,
    device: torch.device,
) -> tuple[float, float]:
    model.train()

    running_loss = 0.0
    running_correct = 0
    running_total = 0

    for images, targets in loader:
        images = images.to(device)
        targets = targets.to(device)

        optimizer.zero_grad()
        logits = model(images)
        loss = criterion(logits, targets)
        loss.backward()
        optimizer.step()

        batch_size = targets.size(0)
        running_loss += loss.item() * batch_size
        running_correct += (logits.argmax(dim=1) == targets).sum().item()
        running_total += batch_size

    epoch_loss = running_loss / running_total
    epoch_acc = running_correct / running_total
    return epoch_loss, epoch_acc


@torch.no_grad()
def validate_one_epoch(
    model: nn.Module,
    loader: DataLoader,
    criterion: nn.Module,
    device: torch.device,
) -> tuple[float, float]:
    model.eval()

    running_loss = 0.0
    running_correct = 0
    running_total = 0

    for images, targets in loader:
        images = images.to(device)
        targets = targets.to(device)

        logits = model(images)
        loss = criterion(logits, targets)

        batch_size = targets.size(0)
        running_loss += loss.item() * batch_size
        running_correct += (logits.argmax(dim=1) == targets).sum().item()
        running_total += batch_size

    epoch_loss = running_loss / running_total
    epoch_acc = running_correct / running_total
    return epoch_loss, epoch_acc


# ---------------------------------------------------------
# 9) Full training routine for one model
# ---------------------------------------------------------
def fit_model(
    model: nn.Module,
    model_name: str,
    train_loader: DataLoader,
    val_loader: DataLoader,
    config: Config,
    device: torch.device,
) -> dict:
    model = model.to(device)

    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=config.lr,
        weight_decay=config.weight_decay,
    )

    history = {
        "model_name": model_name,
        "train_loss": [],
        "val_loss": [],
        "train_acc": [],
        "val_acc": [],
        "best_val_acc": 0.0,
        "best_val_loss": float("inf"),
        "best_epoch": -1,
        "num_parameters": count_trainable_parameters(model),
    }

    best_state_dict = None
    epochs_without_improvement = 0

    save_dir = Path(config.save_dir)
    save_dir.mkdir(parents=True, exist_ok=True)

    print(f"\n=== Training {model_name} ===")
    print(f"Trainable parameters: {history['num_parameters']:,}")

    for epoch in range(1, config.max_epochs + 1):
        train_loss, train_acc = train_one_epoch(
            model, train_loader, criterion, optimizer, device
        )
        val_loss, val_acc = validate_one_epoch(
            model, val_loader, criterion, device
        )

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["train_acc"].append(train_acc)
        history["val_acc"].append(val_acc)

        improved = val_acc > history["best_val_acc"]

        if improved:
            history["best_val_acc"] = val_acc
            history["best_val_loss"] = val_loss
            history["best_epoch"] = epoch
            best_state_dict = copy.deepcopy(model.state_dict())
            epochs_without_improvement = 0

            if config.save_best_weights:
                torch.save(
                    best_state_dict,
                    save_dir / f"{model_name}_best.pt"
                )
        else:
            epochs_without_improvement += 1

        print(
            f"Epoch {epoch:02d}/{config.max_epochs} | "
            f"train_loss={train_loss:.4f} | train_acc={train_acc:.4f} | "
            f"val_loss={val_loss:.4f} | val_acc={val_acc:.4f}"
        )

        if config.use_early_stopping and epochs_without_improvement >= config.early_stopping_patience:
            print(f"Early stopping triggered for {model_name} at epoch {epoch}.")
            break

    # restore best weights into model (optional but useful)
    if best_state_dict is not None:
        model.load_state_dict(best_state_dict)

    history["trained_model"] = model
    return history



# ---------------------------------------------------------
# 10) Train all models
# ---------------------------------------------------------

models = create_model_zoo(
    num_classes=CFG.num_classes,
    in_channels=CFG.in_channels
)

all_histories = {}

for model_name, model in models.items():
    history = fit_model(
        model=model,
        model_name=model_name,
        train_loader=train_loader,
        val_loader=val_loader,
        config=CFG,
        device=DEVICE,
    )
    all_histories[model_name] = history


# ---------------------------------------------------------
# 11) Build comparison table
# ---------------------------------------------------------
def build_comparison_table(histories: dict) -> pd.DataFrame:
    rows = []

    for model_name, hist in histories.items():
        rows.append({
            "model": model_name,
            "num_parameters": hist["num_parameters"],
            "best_epoch": hist["best_epoch"],
            "best_val_acc": hist["best_val_acc"],
            "best_val_loss": hist["best_val_loss"],
            "final_train_acc": hist["train_acc"][-1],
            "final_val_acc": hist["val_acc"][-1],
            "final_train_loss": hist["train_loss"][-1],
            "final_val_loss": hist["val_loss"][-1],
            "acc_gap_final": hist["train_acc"][-1] - hist["val_acc"][-1],
        })

    df = pd.DataFrame(rows)
    df = df.sort_values(by="best_val_acc", ascending=False).reset_index(drop=True)
    return df


comparison_df = build_comparison_table(all_histories)

# ---------------------------------------------------------
# 12) Plot utilities
# ---------------------------------------------------------
def plot_histories(histories: dict) -> None:
    epochs_max = max(len(h["train_loss"]) for h in histories.values())

    # Loss plot
    plt.figure(figsize=(10, 6))
    for model_name, hist in histories.items():
        epochs = range(1, len(hist["train_loss"]) + 1)
        plt.plot(epochs, hist["train_loss"], label=f"{model_name} train")
        plt.plot(epochs, hist["val_loss"], linestyle="--", label=f"{model_name} val")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title("Training and Validation Loss")
    plt.grid(True)
    plt.legend()
    plt.show()

    # Accuracy plot
    plt.figure(figsize=(10, 6))
    for model_name, hist in histories.items():
        epochs = range(1, len(hist["train_acc"]) + 1)
        plt.plot(epochs, hist["train_acc"], label=f"{model_name} train")
        plt.plot(epochs, hist["val_acc"], linestyle="--", label=f"{model_name} val")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.title("Training and Validation Accuracy")
    plt.grid(True)
    plt.legend()
    plt.show()


plot_histories(all_histories)

# ---------------------------------------------------------
# 13) Separate plots per model
# ---------------------------------------------------------
def plot_single_model_history(history: dict) -> None:
    model_name = history["model_name"]
    epochs = range(1, len(history["train_loss"]) + 1)

    plt.figure(figsize=(12, 4))

    plt.subplot(1, 2, 1)
    plt.plot(epochs, history["train_loss"], label="train_loss")
    plt.plot(epochs, history["val_loss"], label="val_loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title(f"{model_name} - Loss")
    plt.grid(True)
    plt.legend()

    plt.subplot(1, 2, 2)
    plt.plot(epochs, history["train_acc"], label="train_acc")
    plt.plot(epochs, history["val_acc"], label="val_acc")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.title(f"{model_name} - Accuracy")
    plt.grid(True)
    plt.legend()

    plt.tight_layout()
    plt.show()


for model_name, history in all_histories.items():
    plot_single_model_history(history)

# ---------------------------------------------------------
# 14) Inspect best model names + saved checkpoints
# ---------------------------------------------------------
best_model_name = comparison_df.iloc[0]["model"]
best_model_history = all_histories[best_model_name]

print("Best model:", best_model_name)
print("Best validation accuracy:", best_model_history["best_val_acc"])
print("Best epoch:", best_model_history["best_epoch"])

if CFG.save_best_weights:
    print("Saved checkpoints in:", Path(CFG.save_dir).resolve())

# ---------------------------------------------------------
# 15) Quick summary text for notebook
# ---------------------------------------------------------
for _, row in comparison_df.iterrows():
    print(
        f"{row['model']}: "
        f"params={int(row['num_parameters']):,}, "
        f"best_val_acc={row['best_val_acc']:.4f}, "
        f"best_epoch={int(row['best_epoch'])}, "
        f"final_gap={row['acc_gap_final']:.4f}"
    )

### Results and interpretation

The results show a clear and consistent performance trend across the four models. Validation accuracy increased from **0.2600** for `ModelA` to **0.3722** for `ModelB`, **0.4998** for `ModelC`, and **0.5688** for `ModelD`. Under the same training conditions, the deeper models therefore achieved substantially better validation performance than the shallower ones.

This supports the expected benefit of depth: with increasing architectural complexity, the models were able to learn richer feature representations and achieved stronger predictive performance on the validation set. In this model zoo, greater depth is also associated with higher capacity and more trainable parameters, so the observed gain should be interpreted as the benefit of **progressively deeper and more expressive CNNs** rather than depth in complete isolation.

Another important observation is that the train-validation gaps remain relatively small after 10 epochs. This suggests that the models are still mostly in the learning phase and are not yet strongly overfitting under this baseline setup. In particular, `ModelD` no longer appears weak once it is given a more reasonable training budget: instead, it becomes the best-performing model in the fair comparison.

Overall, this experiment provides clear evidence for **Objective (a)**: under identical initial training conditions, deeper CNNs performed better on this task than shallower baselines. The next step is therefore to use the stronger models—especially `ModelD`—for more targeted hyperparameter tuning and later regularisation experiments. 

## 3. Objective (b): Hyperparameter Tuning for ModelD

In the fair architecture comparison, `ModelD` achieved the strongest validation performance after 10 epochs and therefore became the main candidate for further optimisation. However, even a strong architecture does not automatically lead to the best possible result. In deep learning, performance depends not only on the model structure itself, but also heavily on how the model is trained.

To address **Objective (b)** of the assignment, I therefore performed a focused hyperparameter sweep for `ModelD`. The aim of this experiment was to show how sensitive the same architecture is to different training settings and to identify a stronger configuration in a systematic way. 

The sweep was intentionally kept compact and interpretable. I varied three important hyperparameters:
- the learning rate,
- the batch size,
- and the weight decay.

All other aspects were kept fixed, including the dataset split, model architecture, optimizer family, image resolution, and the overall training budget of **25 epochs**. This makes the runs directly comparable and allows performance differences to be attributed mainly to the selected hyperparameter settings.

In [ ]:
# HYPERPARAMETER SWEEP FOR ModelD
# =========================================================

from __future__ import annotations

import copy
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader


# ---------------------------------------------------------
# 1) Helper: create loaders for a given batch size
# ---------------------------------------------------------
def make_loaders(batch_size: int):
    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=CFG.num_workers,
        pin_memory=(DEVICE.type == "cuda"),
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=CFG.num_workers,
        pin_memory=(DEVICE.type == "cuda"),
    )

    return train_loader, val_loader


# ---------------------------------------------------------
# 2) Compact search space for ModelD
# Small and reasonable first sweep
# ---------------------------------------------------------
search_space = [
    {"lr": 1e-3, "batch_size": 64, "weight_decay": 0.0},
    {"lr": 3e-4, "batch_size": 64, "weight_decay": 0.0},
    {"lr": 1e-4, "batch_size": 64, "weight_decay": 0.0},
    {"lr": 1e-3, "batch_size": 32, "weight_decay": 0.0},
    {"lr": 3e-4, "batch_size": 32, "weight_decay": 1e-4},
    {"lr": 1e-4, "batch_size": 32, "weight_decay": 1e-4},
]


# ---------------------------------------------------------
# 3) Sweep runner
# ---------------------------------------------------------
def run_modeld_sweep(search_space: list[dict]) -> tuple[dict, pd.DataFrame]:
    sweep_histories = {}
    sweep_rows = []

    for run_idx, params in enumerate(search_space, start=1):
        run_name = (
            f"ModelD_run{run_idx}_"
            f"lr{params['lr']}_"
            f"bs{params['batch_size']}_"
            f"wd{params['weight_decay']}"
        )

        print("=" * 80)
        print(f"Starting run {run_idx}/{len(search_space)}: {run_name}")
        print(params)

        # Fresh config for this run
        cfg_run = copy.deepcopy(CFG)
        cfg_run.lr = params["lr"]
        cfg_run.batch_size = params["batch_size"]
        cfg_run.weight_decay = params["weight_decay"]
        cfg_run.max_epochs = 25
        cfg_run.use_early_stopping = True
        cfg_run.early_stopping_patience = 5
        cfg_run.save_dir = str(Path(CFG.save_dir) / "modeld_sweep")

        # Fresh loaders for batch size
        train_loader_run, val_loader_run = make_loaders(cfg_run.batch_size)

        # Fresh ModelD instance
        modeld = create_model_zoo(
            num_classes=cfg_run.num_classes,
            in_channels=cfg_run.in_channels
        )["ModelD"]

        # Train
        history = fit_model(
            model=modeld,
            model_name=run_name,
            train_loader=train_loader_run,
            val_loader=val_loader_run,
            config=cfg_run,
            device=DEVICE,
        )

        sweep_histories[run_name] = history

        sweep_rows.append({
            "run_name": run_name,
            "lr": params["lr"],
            "batch_size": params["batch_size"],
            "weight_decay": params["weight_decay"],
            "num_parameters": history["num_parameters"],
            "best_epoch": history["best_epoch"],
            "best_val_acc": history["best_val_acc"],
            "best_val_loss": history["best_val_loss"],
            "final_train_acc": history["train_acc"][-1],
            "final_val_acc": history["val_acc"][-1],
            "final_train_loss": history["train_loss"][-1],
            "final_val_loss": history["val_loss"][-1],
            "final_gap": history["train_acc"][-1] - history["val_acc"][-1],
            "epochs_trained": len(history["train_loss"]),
        })

    sweep_df = pd.DataFrame(sweep_rows).sort_values(
        by="best_val_acc",
        ascending=False
    ).reset_index(drop=True)

    return sweep_histories, sweep_df


# ---------------------------------------------------------
# 4) Run the sweep
# ---------------------------------------------------------
modeld_sweep_histories, modeld_sweep_df = run_modeld_sweep(search_space)

modeld_sweep_df.style.format({
    "lr": "{:.0e}",
    "weight_decay": "{:.0e}",
    "best_val_acc": "{:.4f}",
    "best_val_loss": "{:.4f}",
    "final_train_acc": "{:.4f}",
    "final_val_acc": "{:.4f}",
    "final_train_loss": "{:.4f}",
    "final_val_loss": "{:.4f}",
    "final_gap": "{:.4f}",
})

# ---------------------------------------------------------
# 5) Best run summary
# ---------------------------------------------------------
best_run = modeld_sweep_df.iloc[0]

print("Best ModelD sweep run")
print("-" * 40)
print(f"Run name       : {best_run['run_name']}")
print(f"Learning rate  : {best_run['lr']}")
print(f"Batch size     : {best_run['batch_size']}")
print(f"Weight decay   : {best_run['weight_decay']}")
print(f"Best epoch     : {int(best_run['best_epoch'])}")
print(f"Best val acc   : {best_run['best_val_acc']:.4f}")
print(f"Best val loss  : {best_run['best_val_loss']:.4f}")
print(f"Final gap      : {best_run['final_gap']:.4f}")
print(f"Epochs trained : {int(best_run['epochs_trained'])}")

# ---------------------------------------------------------
# 6) Bar plot: best validation accuracy per run
# ---------------------------------------------------------
plt.figure(figsize=(12, 5))
plt.bar(modeld_sweep_df["run_name"], modeld_sweep_df["best_val_acc"])
plt.xticks(rotation=45, ha="right")
plt.ylabel("Best Validation Accuracy")
plt.title("ModelD Hyperparameter Sweep (25 epochs)")
plt.grid(axis="y")
plt.tight_layout()
plt.show()

# ---------------------------------------------------------
# 7) Plot the learning curves of the best run
# ---------------------------------------------------------
best_run_name = best_run["run_name"]
best_history = modeld_sweep_histories[best_run_name]

epochs = range(1, len(best_history["train_loss"]) + 1)

plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(epochs, best_history["train_loss"], label="train_loss")
plt.plot(epochs, best_history["val_loss"], label="val_loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title(f"{best_run_name} - Loss")
plt.grid(True)
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(epochs, best_history["train_acc"], label="train_acc")
plt.plot(epochs, best_history["val_acc"], label="val_acc")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title(f"{best_run_name} - Accuracy")
plt.grid(True)
plt.legend()

plt.tight_layout()
plt.show()

# ---------------------------------------------------------
# 8) Save sweep results as CSV
# ---------------------------------------------------------
results_dir = Path("./results")
results_dir.mkdir(parents=True, exist_ok=True)

csv_path = results_dir / "modeld_hparam_sweep_25epochs.csv"
modeld_sweep_df.to_csv(csv_path, index=False)

print("Saved sweep results to:", csv_path.resolve())

### Results and interpretation

The sweep results show clearly that hyperparameter tuning has a major effect on the performance of `ModelD`. The best configuration was **learning rate = 1e-3**, **batch size = 64**, and **weight decay = 0.0**, reaching a best validation accuracy of **0.6668**. A very similar result was obtained with **learning rate = 1e-3**, **batch size = 32**, and **weight decay = 0.0**, which achieved a best validation accuracy of about **0.6557**.

In contrast, the lower learning-rate configurations performed noticeably worse. The run with **learning rate = 3e-4**, **batch size = 64**, and **weight decay = 0.0** reached about **0.5895**, while the runs with **learning rate = 1e-4** remained substantially below the best configurations. Within this compact search space, the **learning rate** was therefore the most influential hyperparameter.

The tested `weight_decay = 1e-4` settings did not improve validation accuracy in this sweep. This does not mean that weight decay is generally useless, but only that it was not beneficial in the specific configurations tested here. The batch size had some effect, but it was clearly less important than selecting an appropriate learning rate.

The learning curves of the best run also show that the tuned configuration enables effective learning: both training loss and validation loss decrease strongly in the beginning, and validation accuracy rises steadily into the mid-0.66 range. At the same time, the gap between training and validation accuracy becomes more visible towards the end of training, which indicates **moderate overfitting**.

Overall, this experiment directly demonstrates the importance of hyperparameter tuning. The same `ModelD` architecture produced substantially different results depending on the selected training configuration. Therefore, model quality in this task depends not only on architectural depth, but also strongly on choosing suitable hyperparameters.

## 4. Objective (c): Underfitting, Good Fit, and Overfitting

To address **Objective (c)** of the assignment, I designed three controlled experiments to illustrate three different model behaviours: **underfitting**, **good fit**, and **overfitting**. The purpose of this section is not only to compare validation accuracy, but to better understand how model capacity and training data size influence generalisation performance. 

The three experiments were defined as follows:

1. **Underfitting:** `ModelA` was trained on the full training set. As the smallest and least expressive architecture in the model zoo, it serves as a natural candidate for insufficient model capacity.
2. **Good fit / reference:** `ModelD` was trained on the full training set using the strongest visible configuration from the earlier tuning experiments. This represents a higher-capacity model trained under more suitable conditions.
3. **Overfitting:** `ModelD` was trained on only **5% of the training set** (1,200 images) while still being evaluated on the full validation set. This setup was chosen to increase the risk of memorisation and to illustrate how a flexible model can lose generalisation ability when the available training data is too limited.

All three experiments were evaluated on the same validation set of 6,000 images. This makes the learning curves directly comparable and allows the observed behaviour to be interpreted in terms of model complexity, data availability, and generalisation.

In [ ]:
# UNDERFITTING / GOOD FIT / OVERFITTING EXPERIMENTS
# =========================================================

from __future__ import annotations

import copy
import random
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch.utils.data import DataLoader, Subset


# ---------------------------------------------------------
# 1) Helper: reproducible subset selection
# ---------------------------------------------------------
def make_random_subset(dataset, fraction: float, seed: int = 42):
    """
    Create a reproducible random subset from a dataset.
    """
    assert 0 < fraction <= 1.0, "fraction must be in (0, 1]"
    n_total = len(dataset)
    n_subset = max(1, int(n_total * fraction))

    rng = random.Random(seed)
    indices = list(range(n_total))
    rng.shuffle(indices)
    subset_indices = indices[:n_subset]

    return Subset(dataset, subset_indices), subset_indices


# ---------------------------------------------------------
# 2) Helper: DataLoader creation
# ---------------------------------------------------------
def make_loader(dataset, batch_size: int, shuffle: bool):
    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=CFG.num_workers,
        pin_memory=(DEVICE.type == "cuda"),
    )


# ---------------------------------------------------------
# 3) Plot helper for one history
# ---------------------------------------------------------
def plot_single_history(history: dict) -> None:
    model_name = history["model_name"]
    epochs = range(1, len(history["train_loss"]) + 1)

    plt.figure(figsize=(12, 4))

    plt.subplot(1, 2, 1)
    plt.plot(epochs, history["train_loss"], label="train_loss")
    plt.plot(epochs, history["val_loss"], label="val_loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title(f"{model_name} - Loss")
    plt.grid(True)
    plt.legend()

    plt.subplot(1, 2, 2)
    plt.plot(epochs, history["train_acc"], label="train_acc")
    plt.plot(epochs, history["val_acc"], label="val_acc")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.title(f"{model_name} - Accuracy")
    plt.grid(True)
    plt.legend()

    plt.tight_layout()
    plt.show()


# ---------------------------------------------------------
# 4) Plot helper for multiple experiments
# ---------------------------------------------------------
def plot_compare_histories(histories: dict) -> None:
    plt.figure(figsize=(12, 5))
    for exp_name, hist in histories.items():
        epochs = range(1, len(hist["train_loss"]) + 1)
        plt.plot(epochs, hist["train_loss"], label=f"{exp_name} train")
        plt.plot(epochs, hist["val_loss"], linestyle="--", label=f"{exp_name} val")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title("Loss comparison: underfit vs good fit vs overfit")
    plt.grid(True)
    plt.legend()
    plt.show()

    plt.figure(figsize=(12, 5))
    for exp_name, hist in histories.items():
        epochs = range(1, len(hist["train_acc"]) + 1)
        plt.plot(epochs, hist["train_acc"], label=f"{exp_name} train")
        plt.plot(epochs, hist["val_acc"], linestyle="--", label=f"{exp_name} val")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.title("Accuracy comparison: underfit vs good fit vs overfit")
    plt.grid(True)
    plt.legend()
    plt.show()


# ---------------------------------------------------------
# 5) Use the currently best visible ModelD setup as reference
# ---------------------------------------------------------
BEST_VISIBLE_LR = 1e-3
BEST_VISIBLE_BATCH_SIZE = 64
BEST_VISIBLE_WEIGHT_DECAY = 0.0


# ---------------------------------------------------------
# 6) Prepare datasets/loaders
# ---------------------------------------------------------
# Full training set
train_loader_full = make_loader(
    train_dataset,
    batch_size=BEST_VISIBLE_BATCH_SIZE,
    shuffle=True
)

val_loader_full = make_loader(
    val_dataset,
    batch_size=BEST_VISIBLE_BATCH_SIZE,
    shuffle=False
)

# Small subset for overfitting (5% of training set)
train_subset_5, train_subset_indices_5 = make_random_subset(
    train_dataset,
    fraction=0.05,
    seed=CFG.seed
)

train_loader_subset_5 = make_loader(
    train_subset_5,
    batch_size=BEST_VISIBLE_BATCH_SIZE,
    shuffle=True
)

print(f"Full training samples   : {len(train_dataset):,}")
print(f"5% subset train samples: {len(train_subset_5):,}")
print(f"Validation samples      : {len(val_dataset):,}")

# ---------------------------------------------------------
# 7) Define experiment configs
# ---------------------------------------------------------
# Underfitting: ModelA on full training set
cfg_underfit = copy.deepcopy(CFG)
cfg_underfit.lr = BEST_VISIBLE_LR
cfg_underfit.batch_size = BEST_VISIBLE_BATCH_SIZE
cfg_underfit.weight_decay = BEST_VISIBLE_WEIGHT_DECAY
cfg_underfit.max_epochs = 25
cfg_underfit.use_early_stopping = True
cfg_underfit.early_stopping_patience = 5
cfg_underfit.save_dir = str(Path(CFG.save_dir) / "underfit_overfit_study")

# Good fit: ModelD on full training set with current best visible setup
cfg_goodfit = copy.deepcopy(CFG)
cfg_goodfit.lr = BEST_VISIBLE_LR
cfg_goodfit.batch_size = BEST_VISIBLE_BATCH_SIZE
cfg_goodfit.weight_decay = BEST_VISIBLE_WEIGHT_DECAY
cfg_goodfit.max_epochs = 25
cfg_goodfit.use_early_stopping = True
cfg_goodfit.early_stopping_patience = 5
cfg_goodfit.save_dir = str(Path(CFG.save_dir) / "underfit_overfit_study")

# Overfitting: ModelD on only 10% of train set, no early stopping
cfg_overfit = copy.deepcopy(CFG)
cfg_overfit.lr = BEST_VISIBLE_LR
cfg_overfit.batch_size = BEST_VISIBLE_BATCH_SIZE
cfg_overfit.weight_decay = BEST_VISIBLE_WEIGHT_DECAY
cfg_overfit.max_epochs = 60
cfg_overfit.use_early_stopping = False
cfg_overfit.save_dir = str(Path(CFG.save_dir) / "underfit_overfit_study")

# ---------------------------------------------------------
# 8) Run the three experiments
# ---------------------------------------------------------
underfit_overfit_histories = {}

# Experiment 1: Underfitting
underfit_model = ModelA(
    in_channels=cfg_underfit.in_channels,
    num_classes=cfg_underfit.num_classes
)

underfit_history = fit_model(
    model=underfit_model,
    model_name="Underfit_ModelA_full_train",
    train_loader=train_loader_full,
    val_loader=val_loader_full,
    config=cfg_underfit,
    device=DEVICE,
)
underfit_overfit_histories["underfit"] = underfit_history


# Experiment 2: Good fit / reference
goodfit_model = ModelD(
    in_channels=cfg_goodfit.in_channels,
    num_classes=cfg_goodfit.num_classes
)

goodfit_history = fit_model(
    model=goodfit_model,
    model_name="GoodFit_ModelD_full_train",
    train_loader=train_loader_full,
    val_loader=val_loader_full,
    config=cfg_goodfit,
    device=DEVICE,
)
underfit_overfit_histories["good_fit"] = goodfit_history


# Experiment 3: Overfitting
overfit_model = ModelD(
    in_channels=cfg_overfit.in_channels,
    num_classes=cfg_overfit.num_classes
)

overfit_history = fit_model(
    model=overfit_model,
    model_name="Overfit_ModelD_5pct_train",
    train_loader=train_loader_subset_5,
    val_loader=val_loader_full,
    config=cfg_overfit,
    device=DEVICE,
)
underfit_overfit_histories["overfit"] = overfit_history

# ---------------------------------------------------------
# 9) Build comparison table
# ---------------------------------------------------------
def build_underfit_overfit_table(histories: dict) -> pd.DataFrame:
    rows = []

    meta = {
        "underfit": {
            "model": "ModelA",
            "train_fraction": 1.0,
        },
        "good_fit": {
            "model": "ModelD",
            "train_fraction": 1.0,
        },
        "overfit": {
            "model": "ModelD",
            "train_fraction": 0.05,
        },
    }

    for exp_name, hist in histories.items():
        rows.append({
            "experiment": exp_name,
            "model": meta[exp_name]["model"],
            "train_fraction": meta[exp_name]["train_fraction"],
            "best_epoch": hist["best_epoch"],
            "best_val_acc": hist["best_val_acc"],
            "final_train_acc": hist["train_acc"][-1],
            "final_val_acc": hist["val_acc"][-1],
            "final_gap": hist["train_acc"][-1] - hist["val_acc"][-1],
            "num_parameters": hist["num_parameters"],
            "epochs_trained": len(hist["train_loss"]),
        })

    df = pd.DataFrame(rows)
    return df


underfit_overfit_df = build_underfit_overfit_table(underfit_overfit_histories)

# Prettier display
underfit_overfit_df.style.format({
    "train_fraction": "{:.2f}",
    "best_val_acc": "{:.4f}",
    "final_train_acc": "{:.4f}",
    "final_val_acc": "{:.4f}",
    "final_gap": "{:.4f}",
})

# ---------------------------------------------------------
# 10) Plot all experiments together
# ---------------------------------------------------------
plot_compare_histories(underfit_overfit_histories)

# ---------------------------------------------------------
# 11) Plot each experiment separately
# ---------------------------------------------------------
for exp_name, hist in underfit_overfit_histories.items():
    print("=" * 80)
    print(exp_name.upper())
    plot_single_history(hist)

# ---------------------------------------------------------
# 12) Short textual summary in notebook output
# ---------------------------------------------------------
for _, row in underfit_overfit_df.iterrows():
    print(
        f"{row['experiment']}: "
        f"model={row['model']}, "
        f"train_fraction={row['train_fraction']:.2f}, "
        f"best_val_acc={row['best_val_acc']:.4f}, "
        f"final_train_acc={row['final_train_acc']:.4f}, "
        f"final_val_acc={row['final_val_acc']:.4f}, "
        f"final_gap={row['final_gap']:.4f}, "
        f"params={int(row['num_parameters']):,}"
    )

# ---------------------------------------------------------
# 13) Save results table
# ---------------------------------------------------------
results_dir = Path("./results")
results_dir.mkdir(parents=True, exist_ok=True)

csv_path = results_dir / "underfit_goodfit_overfit_results.csv"
underfit_overfit_df.to_csv(csv_path, index=False)

print("Saved results to:", csv_path.resolve())

### Results and interpretation

The three experiments show the expected differences between underfitting, good fit, and overfitting.

In the **underfitting** experiment, `ModelA` achieved only **0.2933** best validation accuracy after 25 epochs. Both training and validation performance remained low throughout the run, and the final train-validation gap stayed very small (about **0.007**). This is a typical sign of underfitting: the model is too simple to learn the task well, so it performs poorly on both the training set and the validation set.

In the **good-fit** reference experiment, `ModelD` trained on the full training set achieved a much stronger best validation accuracy of **0.6747**. The model learned the task effectively, with training accuracy increasing to **0.7993** and validation accuracy to **0.6747**. Although a moderate train-validation gap emerged towards the end, the model still generalised well overall. This makes the experiment a suitable reference for a high-capacity model with good predictive performance under the current setup.

In the **overfitting** experiment, `ModelD` was trained on only **5% of the training set**. Here, the model reached a final training accuracy of **0.5592**, while the final validation accuracy remained at **0.3518**, resulting in a much larger train-validation gap of about **0.207**. The loss curves are particularly informative: after an initial improvement phase, the training loss continued to decrease, while the validation loss started to increase again and reached a much higher level towards the end of training. This is a clear sign of overfitting: the model keeps fitting the limited training subset more closely, but these improvements do not transfer to unseen validation data.

Taken together, these results illustrate the three regimes clearly:
- **underfitting** occurs when model capacity is too low for the task,
- **good fit** occurs when model capacity and available data are reasonably well matched,
- **overfitting** occurs when a flexible model is trained on too little data and begins to specialise too strongly on the training subset.

An additional observation is that the overfitting case is not an extreme memorisation example. Even after 60 epochs, training accuracy is not close to 100%. This suggests that the model is not only affected by data scarcity, but also by the difficulty of the task and the optimisation setup. Nevertheless, the widening train-validation gap and the rising validation loss clearly demonstrate the underlying overfitting behaviour. Overall, this section shows that generalisation depends strongly on the interaction between **model capacity**, **amount of data**, and **training dynamics**. 

## 5. Objective (d) Regularization Experiments for ModelD

After the earlier tuning experiments, `ModelD` emerged as the strongest architecture, but its learning curves also showed clear signs of overfitting. I therefore conducted a dedicated regularization study to examine whether generalization could be improved by adding explicit regularization mechanisms. This directly addresses **Objective (d)** of the assignment. 

The regularization study was revised after follow-up checks showed that the stronger weight decay setting (`1e-4`) was sensitive to the learning rate. To obtain a cleaner comparison, all experiments were repeated for **40 epochs**, and the random seed was reset before each run. The baseline, dropout, and early stopping configurations were trained with `lr = 1e-3`, while the stronger weight decay and combined configurations were trained with `lr = 3e-4`, which had previously been identified as a stable setting. In addition, a new experiment with **mild weight decay (`1e-5`)** was included.

The following configurations were compared on the same `ModelD` backbone and the same train-validation split:
- **baseline** without additional regularization,
- **mild weight decay (`1e-5`)**,
- **stronger weight decay (`1e-4`)**,
- **dropout (`0.2`)** in the classifier head,
- **early stopping**,
- and a **combined** setup using weight decay, dropout, and early stopping.

This design makes the study more interpretable, because differences can be attributed mainly to the tested regularization strategy rather than to uncontrolled training instability.

In [ ]:
# REGULARIZATION EXPERIMENTS FOR ModelD 
# =========================================================

from __future__ import annotations

import copy
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import DataLoader


# ---------------------------------------------------------
# 1) DataLoader helper
# ---------------------------------------------------------
def make_loader(dataset, batch_size: int, shuffle: bool):
    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=CFG.num_workers,
        pin_memory=(DEVICE.type == "cuda"),
    )


# ---------------------------------------------------------
# 2) Reproducibility helper
# ---------------------------------------------------------
def reset_all_seeds(seed: int = 42) -> None:
    set_seed(seed)


# ---------------------------------------------------------
# 3) Dropout variant of ModelD
# We keep the convolutional backbone identical and only
# add dropout in the classifier head.
# ---------------------------------------------------------
class ModelDDropout(nn.Module):
    def __init__(
        self,
        in_channels: int = 3,
        num_classes: int = 10,
        dropout_p: float = 0.2,
    ):
        super().__init__()

        self.features = nn.Sequential(
            DoubleConvBlock(in_channels, 32),
            DoubleConvBlock(32, 64),
            DoubleConvBlock(64, 128),
        )

        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(),
            nn.Linear(128, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(p=dropout_p),
            nn.Linear(128, num_classes),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.features(x)
        x = self.classifier(x)
        return x


# ---------------------------------------------------------
# 4) Shared settings
# ---------------------------------------------------------
REG_BATCH_SIZE = 64
REG_MAX_EPOCHS = 40
REG_PATIENCE = 5

train_loader_full = make_loader(
    train_dataset,
    batch_size=REG_BATCH_SIZE,
    shuffle=True,
)

val_loader_full = make_loader(
    val_dataset,
    batch_size=REG_BATCH_SIZE,
    shuffle=False,
)

print(f"Train samples: {len(train_dataset):,}")
print(f"Val samples  : {len(val_dataset):,}")
print(f"Device       : {DEVICE}")


# ---------------------------------------------------------
# 5) Revised experiment factory
# Notes:
# - baseline / dropout / early_stopping keep lr=1e-3
# - weight decay 1e-4 and combined use lr=3e-4 based on follow-up
# - added an extra smaller weight decay run with 1e-5
# ---------------------------------------------------------
regularization_experiments = {
    "baseline": {
        "model_factory": lambda: ModelD(
            in_channels=CFG.in_channels,
            num_classes=CFG.num_classes,
        ),
        "lr": 1e-3,
        "weight_decay": 0.0,
        "use_early_stopping": False,
        "dropout": 0.0,
        "max_epochs": REG_MAX_EPOCHS,
        "patience": REG_PATIENCE,
        "description": "ModelD without additional regularization",
    },
    "weight_decay_1e-5": {
        "model_factory": lambda: ModelD(
            in_channels=CFG.in_channels,
            num_classes=CFG.num_classes,
        ),
        "lr": 1e-3,
        "weight_decay": 1e-5,
        "use_early_stopping": False,
        "dropout": 0.0,
        "max_epochs": REG_MAX_EPOCHS,
        "patience": REG_PATIENCE,
        "description": "ModelD with mild L2/weight decay",
    },
    "weight_decay_1e-4": {
        "model_factory": lambda: ModelD(
            in_channels=CFG.in_channels,
            num_classes=CFG.num_classes,
        ),
        "lr": 3e-4,
        "weight_decay": 1e-4,
        "use_early_stopping": False,
        "dropout": 0.0,
        "max_epochs": REG_MAX_EPOCHS,
        "patience": REG_PATIENCE,
        "description": "ModelD with stronger L2/weight decay using stable LR",
    },
    "dropout_0.2": {
        "model_factory": lambda: ModelDDropout(
            in_channels=CFG.in_channels,
            num_classes=CFG.num_classes,
            dropout_p=0.2,
        ),
        "lr": 1e-3,
        "weight_decay": 0.0,
        "use_early_stopping": False,
        "dropout": 0.2,
        "max_epochs": REG_MAX_EPOCHS,
        "patience": REG_PATIENCE,
        "description": "ModelD with dropout in classifier head",
    },
    "early_stopping": {
        "model_factory": lambda: ModelD(
            in_channels=CFG.in_channels,
            num_classes=CFG.num_classes,
        ),
        "lr": 1e-3,
        "weight_decay": 0.0,
        "use_early_stopping": True,
        "dropout": 0.0,
        "max_epochs": REG_MAX_EPOCHS,
        "patience": REG_PATIENCE,
        "description": "ModelD with early stopping",
    },
    "combined": {
        "model_factory": lambda: ModelDDropout(
            in_channels=CFG.in_channels,
            num_classes=CFG.num_classes,
            dropout_p=0.2,
        ),
        "lr": 3e-4,
        "weight_decay": 1e-4,
        "use_early_stopping": True,
        "dropout": 0.2,
        "max_epochs": REG_MAX_EPOCHS,
        "patience": REG_PATIENCE,
        "description": "ModelD with combined regularization (weight decay + dropout + early stopping) using stable LR",
    },
}


# ---------------------------------------------------------
# 6) Run all regularization experiments
# ---------------------------------------------------------
regularization_histories = {}
regularization_rows = []

for exp_name, exp in regularization_experiments.items():
    print("=" * 90)
    print(f"Starting regularization experiment: {exp_name}")
    print(exp["description"])

    reset_all_seeds(CFG.seed)

    cfg_run = copy.deepcopy(CFG)
    cfg_run.lr = exp["lr"]
    cfg_run.batch_size = REG_BATCH_SIZE
    cfg_run.weight_decay = exp["weight_decay"]
    cfg_run.max_epochs = exp["max_epochs"]
    cfg_run.use_early_stopping = exp["use_early_stopping"]
    cfg_run.early_stopping_patience = exp["patience"]
    cfg_run.save_dir = str(Path(CFG.save_dir) / "regularization_study_revised_40epochs")

    model = exp["model_factory"]()

    history = fit_model(
        model=model,
        model_name=f"Reg_{exp_name}",
        train_loader=train_loader_full,
        val_loader=val_loader_full,
        config=cfg_run,
        device=DEVICE,
    )

    regularization_histories[exp_name] = history

    regularization_rows.append({
        "experiment": exp_name,
        "description": exp["description"],
        "lr": exp["lr"],
        "weight_decay": exp["weight_decay"],
        "dropout": exp["dropout"],
        "early_stopping": exp["use_early_stopping"],
        "best_epoch": history["best_epoch"],
        "best_val_acc": history["best_val_acc"],
        "best_val_loss": history["best_val_loss"],
        "final_train_acc": history["train_acc"][-1],
        "final_val_acc": history["val_acc"][-1],
        "final_gap": history["train_acc"][-1] - history["val_acc"][-1],
        "num_parameters": history["num_parameters"],
        "epochs_trained": len(history["train_loss"]),
    })

regularization_df = pd.DataFrame(regularization_rows).sort_values(
    by="best_val_acc",
    ascending=False
).reset_index(drop=True)

display(
    regularization_df.style.format({
        "lr": "{:.0e}",
        "weight_decay": "{:.0e}",
        "dropout": "{:.1f}",
        "best_val_acc": "{:.4f}",
        "best_val_loss": "{:.4f}",
        "final_train_acc": "{:.4f}",
        "final_val_acc": "{:.4f}",
        "final_gap": "{:.4f}",
    })
)


# ---------------------------------------------------------
# 7) Plot helper: compare all experiments together
# ---------------------------------------------------------
def plot_regularization_comparison(histories: dict) -> None:
    plt.figure(figsize=(12, 5))
    for exp_name, hist in histories.items():
        epochs = range(1, len(hist["train_loss"]) + 1)
        plt.plot(epochs, hist["train_loss"], label=f"{exp_name} train")
        plt.plot(epochs, hist["val_loss"], linestyle="--", label=f"{exp_name} val")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title("Regularization study - Loss curves")
    plt.grid(True)
    plt.legend()
    plt.show()

    plt.figure(figsize=(12, 5))
    for exp_name, hist in histories.items():
        epochs = range(1, len(hist["train_acc"]) + 1)
        plt.plot(epochs, hist["train_acc"], label=f"{exp_name} train")
        plt.plot(epochs, hist["val_acc"], linestyle="--", label=f"{exp_name} val")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.title("Regularization study - Accuracy curves")
    plt.grid(True)
    plt.legend()
    plt.show()


plot_regularization_comparison(regularization_histories)


# ---------------------------------------------------------
# 8) Plot helper: single experiment
# ---------------------------------------------------------
def plot_single_history(history: dict) -> None:
    model_name = history["model_name"]
    epochs = range(1, len(history["train_loss"]) + 1)

    plt.figure(figsize=(12, 4))

    plt.subplot(1, 2, 1)
    plt.plot(epochs, history["train_loss"], label="train_loss")
    plt.plot(epochs, history["val_loss"], label="val_loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title(f"{model_name} - Loss")
    plt.grid(True)
    plt.legend()

    plt.subplot(1, 2, 2)
    plt.plot(epochs, history["train_acc"], label="train_acc")
    plt.plot(epochs, history["val_acc"], label="val_acc")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.title(f"{model_name} - Accuracy")
    plt.grid(True)
    plt.legend()

    plt.tight_layout()
    plt.show()


for exp_name, history in regularization_histories.items():
    print("=" * 90)
    print(exp_name.upper())
    plot_single_history(history)


# ---------------------------------------------------------
# 9) Best run summary
# ---------------------------------------------------------
best_reg_run = regularization_df.iloc[0]

print("Best regularization run")
print("-" * 40)
print(f"Experiment      : {best_reg_run['experiment']}")
print(f"Description     : {best_reg_run['description']}")
print(f"Learning rate   : {best_reg_run['lr']}")
print(f"Weight decay    : {best_reg_run['weight_decay']}")
print(f"Dropout         : {best_reg_run['dropout']}")
print(f"Early stopping  : {best_reg_run['early_stopping']}")
print(f"Best epoch      : {int(best_reg_run['best_epoch'])}")
print(f"Best val acc    : {best_reg_run['best_val_acc']:.4f}")
print(f"Best val loss   : {best_reg_run['best_val_loss']:.4f}")
print(f"Final train acc : {best_reg_run['final_train_acc']:.4f}")
print(f"Final val acc   : {best_reg_run['final_val_acc']:.4f}")
print(f"Final gap       : {best_reg_run['final_gap']:.4f}")
print(f"Epochs trained  : {int(best_reg_run['epochs_trained'])}")


# ---------------------------------------------------------
# 10) Save results
# ---------------------------------------------------------
results_dir = Path("./results")
results_dir.mkdir(parents=True, exist_ok=True)

csv_path = results_dir / "regularization_results_modeld_revised_40epochs.csv"
regularization_df.to_csv(csv_path, index=False)

print("Saved regularization results to:", csv_path.resolve())

### Results and interpretation

The revised regularization study shows that different regularization methods improved different aspects of training, and that no single method dominated all criteria.

The configuration with the **highest peak validation accuracy** was **dropout (`0.2`)**, which reached **0.6697** at **epoch 33**. This indicates that dropout was the most effective method for maximizing the best observed validation score. However, the same run ended at a clearly lower final validation accuracy (**0.6462**) and still showed a relatively large final train-validation gap (**0.2325**). The learning curves therefore suggest that dropout improved the peak result, but did not fully prevent later overfitting.

The most balanced overall regularization behaviour was obtained with **mild weight decay (`1e-5`)**. This run reached a best validation accuracy of **0.6655** at **epoch 37**, which is only slightly below the dropout peak, while also achieving the **best validation loss (`1.0191`)** and a clearly smaller final train-validation gap (**0.1182**). In addition, its performance remained comparatively stable towards the end of training, finishing at **0.6520** validation accuracy. For this reason, mild weight decay can be interpreted as the strongest regularization choice in terms of overall generalization quality rather than raw peak accuracy.

The **baseline** model remained competitive in terms of peak validation accuracy (**0.6570** at epoch 26), but its learning curves show the clearest overfitting pattern. After the best epoch, the training accuracy continued to increase strongly from about **0.80** to **0.94**, while the validation loss rose sharply from roughly **1.12** to **1.73** by the end of training. This indicates that the unregularized model kept fitting the training data more closely while losing validation performance.

The **early stopping** configuration followed exactly the same trajectory as the baseline until **epoch 31**, matched the same best validation accuracy (**0.6570**), and then stopped before the most severe late-stage deterioration occurred. This means that early stopping did not improve the peak result, but it did provide a useful form of **overfitting control**, since it prevented unnecessary further training once the validation performance had already plateaued or started to degrade.

The stronger regularization settings, **weight decay (`1e-4`)** and the **combined** configuration, are also informative. In the revised setup they no longer collapsed, which confirms that their earlier failure was caused by an unsuitable interaction between regularization strength and learning rate rather than by regularization alone. Nevertheless, both remained clearly below the strongest runs, reaching only **0.6025** and **0.6032** best validation accuracy, respectively. In this study, they therefore appear to be too restrictive.

Overall, the results show that regularization is not automatically beneficial and must itself be tuned carefully. The study also highlights that different regularization methods serve different purposes:
- **dropout (`0.2`)** gave the highest peak validation accuracy,
- **mild weight decay (`1e-5`)** provided the best balance between validation performance, validation loss, and train-validation gap,
- **early stopping** offered the clearest control of late overfitting,
- while **stronger regularization** became trainable after adjustment of the learning rate but remained too strong for the current setup.

In conclusion, this section demonstrates that regularization is a meaningful but highly configuration-dependent design choice. For the present study, **mild weight decay (`1e-5`)** appears to be the most convincing overall regularization strategy, whereas **dropout (`0.2`)** is the best choice if the main objective is to maximize peak validation accuracy.

## 6. Objective (e) Optimizer Comparison for ModelD

To address **Objective (e)** of the assignment, I compared several optimization algorithms on the same `ModelD` architecture. The purpose of this section is to examine how strongly the choice of optimizer influences convergence behaviour, validation performance, and generalization. 

`ModelD` was selected for this study because earlier experiments showed that it is expressive enough to achieve strong results under suitable training conditions. This makes optimizer differences meaningful: when the model itself has sufficient capacity, differences in training dynamics can be attributed more clearly to the optimization algorithm.

To keep the comparison as fair as possible, the following elements were held constant across all runs:
- the train-validation split,
- the model architecture,
- the loss function,
- the batch size,
- the number of training epochs,
- and the weight decay setting.

In addition, early stopping was disabled so that the full learning curves of all optimizers could be compared directly. The compared methods were **Adam**, **RMSprop**, **SGD with momentum**, and **vanilla SGD**.

The learning rates were not forced to be identical across all optimizers. Instead, each method was evaluated with a reasonable optimizer-specific default. This is appropriate because different optimizers use different update mechanisms and often require different step sizes to perform sensibly. The goal of this study is therefore not to construct an artificially identical setup, but to compare the optimizers under practical and interpretable conditions.

In [ ]:
# OPTIMIZER COMPARISON FOR ModelD
# =========================================================
from __future__ import annotations

import copy
import time
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import DataLoader


# ---------------------------------------------------------
# 1) DataLoader helper
# ---------------------------------------------------------
def make_loader(dataset, batch_size: int, shuffle: bool):
    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=CFG.num_workers,
        pin_memory=(DEVICE.type == "cuda"),
    )


# ---------------------------------------------------------
# 2) Shared setup for a fair optimizer comparison
# Keep model/data/loss/batch size/epochs fixed
# ---------------------------------------------------------
OPT_BATCH_SIZE = 64
OPT_MAX_EPOCHS = 25
OPT_WEIGHT_DECAY = 0.0   # keep fixed for fairness
OPT_USE_EARLY_STOPPING = False  # better for full curve comparison

train_loader_full = make_loader(
    train_dataset,
    batch_size=OPT_BATCH_SIZE,
    shuffle=True,
)

val_loader_full = make_loader(
    val_dataset,
    batch_size=OPT_BATCH_SIZE,
    shuffle=False,
)

print(f"Train samples: {len(train_dataset):,}")
print(f"Val samples  : {len(val_dataset):,}")
print(f"Device       : {DEVICE}")

# ---------------------------------------------------------
# 3) Train / validate helpers
# ---------------------------------------------------------
def count_trainable_parameters(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


def train_one_epoch(
    model: nn.Module,
    loader: DataLoader,
    criterion: nn.Module,
    optimizer: torch.optim.Optimizer,
    device: torch.device,
):
    model.train()

    running_loss = 0.0
    running_correct = 0
    running_total = 0

    for images, targets in loader:
        images = images.to(device)
        targets = targets.to(device)

        optimizer.zero_grad()
        logits = model(images)
        loss = criterion(logits, targets)
        loss.backward()
        optimizer.step()

        batch_size = targets.size(0)
        running_loss += loss.item() * batch_size
        running_correct += (logits.argmax(dim=1) == targets).sum().item()
        running_total += batch_size

    epoch_loss = running_loss / running_total
    epoch_acc = running_correct / running_total
    return epoch_loss, epoch_acc


@torch.no_grad()
def validate_one_epoch(
    model: nn.Module,
    loader: DataLoader,
    criterion: nn.Module,
    device: torch.device,
):
    model.eval()

    running_loss = 0.0
    running_correct = 0
    running_total = 0

    for images, targets in loader:
        images = images.to(device)
        targets = targets.to(device)

        logits = model(images)
        loss = criterion(logits, targets)

        batch_size = targets.size(0)
        running_loss += loss.item() * batch_size
        running_correct += (logits.argmax(dim=1) == targets).sum().item()
        running_total += batch_size

    epoch_loss = running_loss / running_total
    epoch_acc = running_correct / running_total
    return epoch_loss, epoch_acc

# ---------------------------------------------------------
# 4) Generic fit function with configurable optimizer
# ---------------------------------------------------------
def fit_model_with_optimizer(
    model: nn.Module,
    model_name: str,
    train_loader: DataLoader,
    val_loader: DataLoader,
    config,
    optimizer_name: str,
    optimizer_factory,
    device: torch.device,
):
    model = model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optimizer_factory(model.parameters())

    history = {
        "model_name": model_name,
        "optimizer_name": optimizer_name,
        "train_loss": [],
        "val_loss": [],
        "train_acc": [],
        "val_acc": [],
        "best_val_acc": 0.0,
        "best_val_loss": float("inf"),
        "best_epoch": -1,
        "num_parameters": count_trainable_parameters(model),
        "epoch_times_sec": [],
    }

    best_state_dict = None

    save_dir = Path(config.save_dir)
    save_dir.mkdir(parents=True, exist_ok=True)

    print(f"\n=== Training {model_name} ===")
    print(f"Optimizer           : {optimizer_name}")
    print(f"Trainable parameters: {history['num_parameters']:,}")

    for epoch in range(1, config.max_epochs + 1):
        t0 = time.perf_counter()

        train_loss, train_acc = train_one_epoch(
            model, train_loader, criterion, optimizer, device
        )
        val_loss, val_acc = validate_one_epoch(
            model, val_loader, criterion, device
        )

        dt = time.perf_counter() - t0

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["train_acc"].append(train_acc)
        history["val_acc"].append(val_acc)
        history["epoch_times_sec"].append(dt)

        if val_acc > history["best_val_acc"]:
            history["best_val_acc"] = val_acc
            history["best_val_loss"] = val_loss
            history["best_epoch"] = epoch
            best_state_dict = copy.deepcopy(model.state_dict())

            torch.save(
                best_state_dict,
                save_dir / f"{model_name}_best.pt"
            )

        print(
            f"Epoch {epoch:02d}/{config.max_epochs} | "
            f"train_loss={train_loss:.4f} | train_acc={train_acc:.4f} | "
            f"val_loss={val_loss:.4f} | val_acc={val_acc:.4f} | "
            f"time={dt:.2f}s"
        )

    if best_state_dict is not None:
        model.load_state_dict(best_state_dict)

    history["trained_model"] = model
    return history

# ---------------------------------------------------------
# 5) Optimizer experiment definitions
# Use reasonable optimizer-specific defaults
# ---------------------------------------------------------
optimizer_experiments = {
    "adam": {
        "optimizer_name": "Adam",
        "optimizer_factory": lambda params: torch.optim.Adam(
            params,
            lr=1e-3,
            weight_decay=OPT_WEIGHT_DECAY,
        ),
        "description": "Adam with lr=1e-3",
    },
    "rmsprop": {
        "optimizer_name": "RMSprop",
        "optimizer_factory": lambda params: torch.optim.RMSprop(
            params,
            lr=1e-3,
            alpha=0.99,
            weight_decay=OPT_WEIGHT_DECAY,
        ),
        "description": "RMSprop with lr=1e-3, alpha=0.99",
    },
    "sgd_momentum": {
        "optimizer_name": "SGD+Momentum",
        "optimizer_factory": lambda params: torch.optim.SGD(
            params,
            lr=1e-2,
            momentum=0.9,
            weight_decay=OPT_WEIGHT_DECAY,
            nesterov=False,
        ),
        "description": "SGD with lr=1e-2, momentum=0.9",
    },
    # Optional baseline
    "sgd": {
        "optimizer_name": "SGD",
        "optimizer_factory": lambda params: torch.optim.SGD(
            params,
            lr=1e-2,
            momentum=0.0,
            weight_decay=OPT_WEIGHT_DECAY,
        ),
        "description": "Vanilla SGD with lr=1e-2",
    },
}

# ---------------------------------------------------------
# 6) Run all optimizer experiments on the same model
# ---------------------------------------------------------
cfg_opt = copy.deepcopy(CFG)
cfg_opt.batch_size = OPT_BATCH_SIZE
cfg_opt.max_epochs = OPT_MAX_EPOCHS
cfg_opt.weight_decay = OPT_WEIGHT_DECAY
cfg_opt.use_early_stopping = OPT_USE_EARLY_STOPPING
cfg_opt.save_dir = str(Path(CFG.save_dir) / "optimizer_study")

optimizer_histories = {}
optimizer_rows = []

for exp_name, exp in optimizer_experiments.items():
    print("=" * 90)
    print(f"Starting optimizer experiment: {exp_name}")
    print(exp["description"])

    model = ModelD(
        in_channels=cfg_opt.in_channels,
        num_classes=cfg_opt.num_classes,
    )

    history = fit_model_with_optimizer(
        model=model,
        model_name=f"Opt_{exp_name}",
        train_loader=train_loader_full,
        val_loader=val_loader_full,
        config=cfg_opt,
        optimizer_name=exp["optimizer_name"],
        optimizer_factory=exp["optimizer_factory"],
        device=DEVICE,
    )

    optimizer_histories[exp_name] = history

    optimizer_rows.append({
        "experiment": exp_name,
        "optimizer": exp["optimizer_name"],
        "description": exp["description"],
        "best_epoch": history["best_epoch"],
        "best_val_acc": history["best_val_acc"],
        "best_val_loss": history["best_val_loss"],
        "final_train_acc": history["train_acc"][-1],
        "final_val_acc": history["val_acc"][-1],
        "final_gap": history["train_acc"][-1] - history["val_acc"][-1],
        "avg_epoch_time_sec": sum(history["epoch_times_sec"]) / len(history["epoch_times_sec"]),
        "num_parameters": history["num_parameters"],
        "epochs_trained": len(history["train_loss"]),
    })

optimizer_df = pd.DataFrame(optimizer_rows).sort_values(
    by="best_val_acc",
    ascending=False
).reset_index(drop=True)

# Prettier display
optimizer_df.style.format({
    "best_val_acc": "{:.4f}",
    "best_val_loss": "{:.4f}",
    "final_train_acc": "{:.4f}",
    "final_val_acc": "{:.4f}",
    "final_gap": "{:.4f}",
    "avg_epoch_time_sec": "{:.2f}",
})

# ---------------------------------------------------------
# 7) Plot comparison of all optimizers
# ---------------------------------------------------------
def plot_optimizer_comparison(histories: dict) -> None:
    plt.figure(figsize=(12, 5))
    for exp_name, hist in histories.items():
        epochs = range(1, len(hist["train_loss"]) + 1)
        plt.plot(epochs, hist["train_loss"], label=f"{exp_name} train")
        plt.plot(epochs, hist["val_loss"], linestyle="--", label=f"{exp_name} val")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title("Optimizer comparison - Loss curves")
    plt.grid(True)
    plt.legend()
    plt.show()

    plt.figure(figsize=(12, 5))
    for exp_name, hist in histories.items():
        epochs = range(1, len(hist["train_acc"]) + 1)
        plt.plot(epochs, hist["train_acc"], label=f"{exp_name} train")
        plt.plot(epochs, hist["val_acc"], linestyle="--", label=f"{exp_name} val")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.title("Optimizer comparison - Accuracy curves")
    plt.grid(True)
    plt.legend()
    plt.show()


plot_optimizer_comparison(optimizer_histories)

# ---------------------------------------------------------
# 8) Single-run plots
# ---------------------------------------------------------
def plot_single_history(history: dict) -> None:
    model_name = history["model_name"]
    epochs = range(1, len(history["train_loss"]) + 1)

    plt.figure(figsize=(12, 4))

    plt.subplot(1, 2, 1)
    plt.plot(epochs, history["train_loss"], label="train_loss")
    plt.plot(epochs, history["val_loss"], label="val_loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title(f"{model_name} - Loss")
    plt.grid(True)
    plt.legend()

    plt.subplot(1, 2, 2)
    plt.plot(epochs, history["train_acc"], label="train_acc")
    plt.plot(epochs, history["val_acc"], label="val_acc")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.title(f"{model_name} - Accuracy")
    plt.grid(True)
    plt.legend()

    plt.tight_layout()
    plt.show()


for exp_name, history in optimizer_histories.items():
    print("=" * 90)
    print(exp_name.upper())
    plot_single_history(history)

# ---------------------------------------------------------
# 9) Best optimizer summary
# ---------------------------------------------------------
best_opt_run = optimizer_df.iloc[0]

print("Best optimizer run")
print("-" * 40)
print(f"Experiment         : {best_opt_run['experiment']}")
print(f"Optimizer          : {best_opt_run['optimizer']}")
print(f"Description        : {best_opt_run['description']}")
print(f"Best epoch         : {int(best_opt_run['best_epoch'])}")
print(f"Best val acc       : {best_opt_run['best_val_acc']:.4f}")
print(f"Best val loss      : {best_opt_run['best_val_loss']:.4f}")
print(f"Final train acc    : {best_opt_run['final_train_acc']:.4f}")
print(f"Final val acc      : {best_opt_run['final_val_acc']:.4f}")
print(f"Final gap          : {best_opt_run['final_gap']:.4f}")
print(f"Avg epoch time [s] : {best_opt_run['avg_epoch_time_sec']:.2f}")
print(f"Epochs trained     : {int(best_opt_run['epochs_trained'])}")

# ---------------------------------------------------------
# 10) Save results
# ---------------------------------------------------------
results_dir = Path("./results")
results_dir.mkdir(parents=True, exist_ok=True)

csv_path = results_dir / "optimizer_comparison_modeld.csv"
optimizer_df.to_csv(csv_path, index=False)

print("Saved optimizer results to:", csv_path.resolve())

### Results and interpretation

The optimizer comparison shows that the choice of optimization algorithm has a substantial effect on the training dynamics and final performance of `ModelD`.

Among the tested methods, **RMSprop** achieved the strongest result, reaching a best validation accuracy of **0.6727** at **epoch 25**. Its learning curves show that it improved steadily throughout training and slightly outperformed Adam in the later phase of optimization. RMSprop therefore appears to be the most effective optimizer in this study in terms of peak validation performance.

**Adam** was the second-best optimizer, reaching a best validation accuracy of **0.6567** at **epoch 25**. It learned stably and competitively throughout the full training run, and its overall behaviour was strong and reliable. However, its final validation performance remained below RMSprop, especially in the later epochs.

**SGD with momentum** eventually learned the task, but much more slowly. It reached only **0.4320** validation accuracy at **epoch 25**, which is clearly below the adaptive optimizers. The learning curves indicate that the optimizer needed many epochs before making substantial progress, suggesting that the chosen setup was less suitable for efficient convergence.

The weakest result was obtained with **vanilla SGD**, which stayed essentially at chance level, around **0.10** validation accuracy. Under the present configuration, it did not become competitive at all. This does not necessarily mean that SGD is universally ineffective for this problem. A more careful tuning procedure, for example with a different learning rate, learning-rate schedule, or larger training budget, might lead to better results. However, within the current setup, vanilla SGD was clearly not suitable.

The measured epoch times were broadly similar across optimizers, with the SGD variants being slightly faster on average. Nevertheless, this small computational advantage did not compensate for their much weaker predictive performance. In practical terms, the adaptive optimizers offered a much better trade-off between convergence and accuracy.

Overall, this experiment shows that optimizer choice is not merely a technical detail. Even when the model, data, and training budget are kept fixed, different optimizers can lead to markedly different outcomes. In this study, **RMSprop** was the strongest optimizer overall, **Adam** was a competitive second choice, **SGD with momentum** learned but remained clearly weaker, and **vanilla SGD** was not competitive under the chosen configuration. This confirms the importance of comparing optimization algorithms in deep learning experiments. 

## 7. Optional Objective: Advanced Architectures Compared with a Simpler Baseline

To extend the study beyond the custom CNN architectures, I compared the strongest simple baseline (`ModelD`) with two well-known deep vision architectures: **ResNet18** and **DenseNet121**. The purpose of this section is to investigate whether more advanced architectures provide a measurable advantage over a simpler custom model on the iCoSimal V3 classification task. This addresses the optional objective of comparing advanced architectures with simpler ones. 

To keep the comparison meaningful, all three models were trained under the same general setup: the same train-validation split, loss function, batch size, learning rate, number of epochs, and `weight_decay = 0.0` were used throughout. In addition, early stopping was disabled so that the full learning curves could be compared directly.

An important design choice is that **all advanced architectures were trained from scratch** (`weights=None`). This means that the comparison focuses on architectural differences alone and does not include any transfer-learning advantage from pre-trained weights. As a result, the section evaluates whether architectural sophistication by itself leads to better performance under the current training budget.

In [ ]:
# ADVANCED ARCHITECTURES COMPARISON
# Compare:
# - ModelD (simple/custom baseline)
# - ResNet18
# - DenseNet121
# =========================================================

from __future__ import annotations

import copy
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import models


# ---------------------------------------------------------
# 1) DataLoader helper
# ---------------------------------------------------------
def make_loader(dataset, batch_size: int, shuffle: bool):
    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=CFG.num_workers,
        pin_memory=(DEVICE.type == "cuda"),
    )

# ---------------------------------------------------------
# 2) Model factories
# Train from scratch: weights=None
# ---------------------------------------------------------
def create_modeld(num_classes: int, in_channels: int = 3):
    return ModelD(
        in_channels=in_channels,
        num_classes=num_classes,
    )


def create_resnet18(num_classes: int):
    model = models.resnet18(weights=None)
    in_features = model.fc.in_features
    model.fc = nn.Linear(in_features, num_classes)
    return model


def create_densenet121(num_classes: int):
    model = models.densenet121(weights=None)
    in_features = model.classifier.in_features
    model.classifier = nn.Linear(in_features, num_classes)
    return model

# ---------------------------------------------------------
# 3) Shared training setup
# Keep the comparison fair and compact
# ---------------------------------------------------------
ADV_BATCH_SIZE = 64
ADV_MAX_EPOCHS = 25
ADV_LR = 1e-3
ADV_WEIGHT_DECAY = 0.0
ADV_USE_EARLY_STOPPING = False  # full curves for fair comparison

train_loader_adv = make_loader(
    train_dataset,
    batch_size=ADV_BATCH_SIZE,
    shuffle=True,
)

val_loader_adv = make_loader(
    val_dataset,
    batch_size=ADV_BATCH_SIZE,
    shuffle=False,
)

print(f"Train samples: {len(train_dataset):,}")
print(f"Val samples  : {len(val_dataset):,}")
print(f"Device       : {DEVICE}")

# ---------------------------------------------------------
# 4) Architecture experiments
# ---------------------------------------------------------
advanced_architecture_experiments = {
    "ModelD": {
        "model_factory": lambda: create_modeld(
            num_classes=CFG.num_classes,
            in_channels=CFG.in_channels,
        ),
        "description": "Custom simple CNN baseline",
    },
    "ResNet18": {
        "model_factory": lambda: create_resnet18(
            num_classes=CFG.num_classes,
        ),
        "description": "Residual network (ResNet18), from scratch",
    },
    "DenseNet121": {
        "model_factory": lambda: create_densenet121(
            num_classes=CFG.num_classes,
        ),
        "description": "Dense connectivity network (DenseNet121), from scratch",
    },
}

# ---------------------------------------------------------
# 5) Run all architecture experiments
# Uses your existing fit_model(...)
# ---------------------------------------------------------
advanced_histories = {}
advanced_rows = []

for arch_name, exp in advanced_architecture_experiments.items():
    print("=" * 90)
    print(f"Starting advanced architecture experiment: {arch_name}")
    print(exp["description"])

    cfg_run = copy.deepcopy(CFG)
    cfg_run.lr = ADV_LR
    cfg_run.batch_size = ADV_BATCH_SIZE
    cfg_run.weight_decay = ADV_WEIGHT_DECAY
    cfg_run.max_epochs = ADV_MAX_EPOCHS
    cfg_run.use_early_stopping = ADV_USE_EARLY_STOPPING
    cfg_run.save_dir = str(Path(CFG.save_dir) / "advanced_architecture_study")

    model = exp["model_factory"]()

    history = fit_model(
        model=model,
        model_name=f"AdvArch_{arch_name}",
        train_loader=train_loader_adv,
        val_loader=val_loader_adv,
        config=cfg_run,
        device=DEVICE,
    )

    advanced_histories[arch_name] = history

    advanced_rows.append({
        "architecture": arch_name,
        "description": exp["description"],
        "best_epoch": history["best_epoch"],
        "best_val_acc": history["best_val_acc"],
        "best_val_loss": history["best_val_loss"],
        "final_train_acc": history["train_acc"][-1],
        "final_val_acc": history["val_acc"][-1],
        "final_gap": history["train_acc"][-1] - history["val_acc"][-1],
        "num_parameters": history["num_parameters"],
        "epochs_trained": len(history["train_loss"]),
    })

advanced_df = pd.DataFrame(advanced_rows).sort_values(
    by="best_val_acc",
    ascending=False
).reset_index(drop=True)

# Prettier notebook display
display(
    advanced_df.style.format({
        "best_val_acc": "{:.4f}",
        "best_val_loss": "{:.4f}",
        "final_train_acc": "{:.4f}",
        "final_val_acc": "{:.4f}",
        "final_gap": "{:.4f}",
    })
)

# ---------------------------------------------------------
# 6) Plot helper: compare all architectures together
# ---------------------------------------------------------
def plot_advanced_architecture_comparison(histories: dict) -> None:
    plt.figure(figsize=(12, 5))
    for arch_name, hist in histories.items():
        epochs = range(1, len(hist["train_loss"]) + 1)
        plt.plot(epochs, hist["train_loss"], label=f"{arch_name} train")
        plt.plot(epochs, hist["val_loss"], linestyle="--", label=f"{arch_name} val")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title("Advanced architectures - Loss curves")
    plt.grid(True)
    plt.legend()
    plt.show()

    plt.figure(figsize=(12, 5))
    for arch_name, hist in histories.items():
        epochs = range(1, len(hist["train_acc"]) + 1)
        plt.plot(epochs, hist["train_acc"], label=f"{arch_name} train")
        plt.plot(epochs, hist["val_acc"], linestyle="--", label=f"{arch_name} val")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.title("Advanced architectures - Accuracy curves")
    plt.grid(True)
    plt.legend()
    plt.show()


plot_advanced_architecture_comparison(advanced_histories)

# ---------------------------------------------------------
# 7) Single-run plots
# ---------------------------------------------------------
def plot_single_history(history: dict) -> None:
    model_name = history["model_name"]
    epochs = range(1, len(history["train_loss"]) + 1)

    plt.figure(figsize=(12, 4))

    plt.subplot(1, 2, 1)
    plt.plot(epochs, history["train_loss"], label="train_loss")
    plt.plot(epochs, history["val_loss"], label="val_loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title(f"{model_name} - Loss")
    plt.grid(True)
    plt.legend()

    plt.subplot(1, 2, 2)
    plt.plot(epochs, history["train_acc"], label="train_acc")
    plt.plot(epochs, history["val_acc"], label="val_acc")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.title(f"{model_name} - Accuracy")
    plt.grid(True)
    plt.legend()

    plt.tight_layout()
    plt.show()


for arch_name, history in advanced_histories.items():
    print("=" * 90)
    print(arch_name.upper())
    plot_single_history(history)

# ---------------------------------------------------------
# 8) Best run summary
# ---------------------------------------------------------
best_adv_run = advanced_df.iloc[0]

print("Best advanced architecture run")
print("-" * 40)
print(f"Architecture     : {best_adv_run['architecture']}")
print(f"Description      : {best_adv_run['description']}")
print(f"Best epoch       : {int(best_adv_run['best_epoch'])}")
print(f"Best val acc     : {best_adv_run['best_val_acc']:.4f}")
print(f"Best val loss    : {best_adv_run['best_val_loss']:.4f}")
print(f"Final train acc  : {best_adv_run['final_train_acc']:.4f}")
print(f"Final val acc    : {best_adv_run['final_val_acc']:.4f}")
print(f"Final gap        : {best_adv_run['final_gap']:.4f}")
print(f"Parameters       : {int(best_adv_run['num_parameters']):,}")
print(f"Epochs trained   : {int(best_adv_run['epochs_trained'])}")

# ---------------------------------------------------------
# 9) Save results
# ---------------------------------------------------------
results_dir = Path("./results")
results_dir.mkdir(parents=True, exist_ok=True)

csv_path = results_dir / "advanced_architectures_results.csv"
advanced_df.to_csv(csv_path, index=False)

print("Saved advanced architecture results to:", csv_path.resolve())

### Results and interpretation

The comparison shows that more advanced architectures do not automatically dominate a simpler custom CNN when all models are trained from scratch under the same training budget.

Among the tested models, **DenseNet121** achieved the highest **peak validation accuracy**, reaching **0.6740** at **epoch 12**. This indicates that the dense connectivity pattern can provide a strong representational advantage on this task. However, the learning curves also show that DenseNet121 became increasingly unstable afterwards: training accuracy continued to rise to about **0.98**, while validation loss increased substantially and validation performance fluctuated. This suggests that the model has very high capacity, but also a stronger tendency to overfit under the current setup.

**ModelD** reached a best validation accuracy of **0.6528** at **epoch 25**, which is below DenseNet121 in peak performance but still competitive. Its main strength is that it learned more steadily and remained considerably more stable across the full training run. In addition, it achieved this result with only **304,810** trainable parameters, compared with **6,964,106** for DenseNet121 and **11,181,642** for ResNet18. This makes `ModelD` by far the most parameter-efficient architecture in the comparison.

**ResNet18** achieved a best validation accuracy of **0.6222** at **epoch 10**, which is weaker than both DenseNet121 and ModelD. Similar to DenseNet121, ResNet18 reached extremely high training accuracy, again close to **0.98**, but this did not translate into superior validation performance. The model therefore appears to overfit strongly in this setting and does not provide a practical advantage over the simpler baseline.

A useful interpretation is to distinguish between three criteria:
- **Peak validation performance:** best for **DenseNet121**,
- **Stability and robustness across training:** strongest for **ModelD**,
- **Parameter efficiency:** clearly best for **ModelD**.

These results show that advanced architectures trained from scratch can offer benefits, but they also introduce higher capacity and a stronger risk of overfitting. In this study, DenseNet121 slightly outperformed the simpler baseline in terms of best validation accuracy, but ModelD remained highly competitive while being much smaller and more stable. ResNet18, in contrast, did not surpass the custom baseline.

Overall, this section shows that architectural sophistication alone is not a guarantee of better results. Under the present setup, **DenseNet121** delivered the best peak score, but **ModelD** provided the most balanced trade-off between performance, stability, and parameter efficiency. 

## 8. Optional Objective: Monitoring and Automated Hyperparameter Optimization with W&B and Optuna

To extend the manual tuning experiments, I combined **Weights & Biases (W&B)** for experiment tracking with **Optuna** for automated hyperparameter optimization on `ModelD`. This addresses two optional objectives of the assignment: monitoring the training process and using a framework to automatically search for strong hyperparameter configurations. 

The purpose of this section is twofold. First, W&B provides a structured way to monitor training and validation curves across runs, making it easier to compare learning dynamics, detect unstable configurations, and document the optimization process. Second, Optuna enables a more systematic search over the hyperparameter space than manual trial-and-error.

The search space included the optimizer type, batch size, dropout rate, learning rate, and weight decay. To keep the search computationally manageable, each Optuna trial was intentionally limited to **12 epochs**. After identifying the most promising configuration, the best Optuna setup was retrained for **25 epochs** in order to evaluate its full performance under a more realistic training budget.

In [ ]:
# Optuna & W&B integration for Hyperparameter Optimization (HPO)
# =========================================================
from __future__ import annotations

import copy
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import DataLoader

import wandb
import optuna
from optuna.exceptions import TrialPruned

wandb.login()  # interactive in notebook

# =========================================================
# 2) Project settings
# =========================================================
WANDB_PROJECT = "***ModelD_HPO"  # change this to your desired W&B project name
WANDB_ENTITY = "***"  # change this to your W&B username or team name

HPO_RESULTS_DIR = Path("./results")
HPO_RESULTS_DIR.mkdir(parents=True, exist_ok=True)

OPTUNA_STORAGE = f"sqlite:///{(HPO_RESULTS_DIR / 'optuna_modeld_wandb.db').resolve()}"

# =========================================================
# 3) DataLoader helper
# =========================================================
def make_loader(dataset, batch_size: int, shuffle: bool):
    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=CFG.num_workers,
        pin_memory=(DEVICE.type == "cuda"),
    )

# =========================================================
# 4) Optional dropout variant of ModelD for HPO
# If dropout=0.0, we use the original ModelD.
# If dropout>0.0, we use a classifier-head dropout variant.
# =========================================================
class ModelDDropoutHPO(nn.Module):
    def __init__(
        self,
        in_channels: int = 3,
        num_classes: int = 10,
        dropout_p: float = 0.2,
    ):
        super().__init__()
        self.features = nn.Sequential(
            DoubleConvBlock(in_channels, 32),
            DoubleConvBlock(32, 64),
            DoubleConvBlock(64, 128),
        )
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(),
            nn.Linear(128, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(p=dropout_p),
            nn.Linear(128, num_classes),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.features(x)
        x = self.classifier(x)
        return x


def build_model_for_config(dropout: float) -> nn.Module:
    if dropout <= 0.0:
        return ModelD(
            in_channels=CFG.in_channels,
            num_classes=CFG.num_classes,
        )
    return ModelDDropoutHPO(
        in_channels=CFG.in_channels,
        num_classes=CFG.num_classes,
        dropout_p=dropout,
    )

# =========================================================
# 5) Robust train/validate helpers
# Fresh implementation to avoid relying on any previously
# overwritten notebook functions.
# =========================================================
def train_one_epoch(
    model: nn.Module,
    loader,
    criterion: nn.Module,
    optimizer: torch.optim.Optimizer,
    device: torch.device,
):
    model.train()

    running_loss = 0.0
    running_correct = 0
    running_total = 0

    for images, targets in loader:
        images = images.to(device)
        targets = targets.to(device)

        optimizer.zero_grad()
        logits = model(images)
        loss = criterion(logits, targets)
        loss.backward()
        optimizer.step()

        batch_size = targets.size(0)
        running_loss += loss.item() * batch_size
        running_correct += (logits.argmax(dim=1) == targets).sum().item()
        running_total += batch_size

    return running_loss / running_total, running_correct / running_total


@torch.no_grad()
def validate_one_epoch(
    model: nn.Module,
    loader,
    criterion: nn.Module,
    device: torch.device,
):
    model.eval()

    running_loss = 0.0
    running_correct = 0
    running_total = 0

    for images, targets in loader:
        images = images.to(device)
        targets = targets.to(device)

        logits = model(images)
        loss = criterion(logits, targets)

        batch_size = targets.size(0)
        running_loss += loss.item() * batch_size
        running_correct += (logits.argmax(dim=1) == targets).sum().item()
        running_total += batch_size

    return running_loss / running_total, running_correct / running_total

# =========================================================
# 6) Optimizer builder
# Conditional search space = good Optuna practice
# =========================================================
def build_optimizer(
    optimizer_name: str,
    model: nn.Module,
    lr: float,
    weight_decay: float,
):
    if optimizer_name == "adam":
        return torch.optim.Adam(
            model.parameters(),
            lr=lr,
            weight_decay=weight_decay,
        )
    elif optimizer_name == "rmsprop":
        return torch.optim.RMSprop(
            model.parameters(),
            lr=lr,
            alpha=0.99,
            weight_decay=weight_decay,
        )
    elif optimizer_name == "sgd_momentum":
        return torch.optim.SGD(
            model.parameters(),
            lr=lr,
            momentum=0.9,
            weight_decay=weight_decay,
            nesterov=False,
        )
    else:
        raise ValueError(f"Unknown optimizer_name: {optimizer_name}")
    
# =========================================================
# 7) One manually chosen W&B-monitored baseline run
# This run primarily demonstrates W&B-based monitoring on a manually chosen setup.
# It is a short reference run for curve inspection, not the main manual performance
# baseline for the final manual-vs-Optuna comparison.
# Update MANUAL_BEST_VAL_ACC if your best manual run changes.
# =========================================================
MANUAL_BASELINE_CONFIG = {
    "optimizer": "adam",
    "lr": 1e-3,
    "weight_decay": 0.0,
    "batch_size": 64,
    "dropout": 0.0,
    "epochs": 8,
}

MANUAL_BEST_VAL_ACC = 0.6668  # update if you have a newer best manual result


def run_wandb_baseline(config: dict):
    train_loader = make_loader(train_dataset, config["batch_size"], shuffle=True)
    val_loader = make_loader(val_dataset, config["batch_size"], shuffle=False)

    model = build_model_for_config(config["dropout"]).to(DEVICE)
    criterion = nn.CrossEntropyLoss()
    optimizer = build_optimizer(
        optimizer_name=config["optimizer"],
        model=model,
        lr=config["lr"],
        weight_decay=config["weight_decay"],
    )

    history = {
        "train_loss": [],
        "val_loss": [],
        "train_acc": [],
        "val_acc": [],
    }

    with wandb.init(
        project=WANDB_PROJECT,
        entity=WANDB_ENTITY,
        config=config,
        job_type="manual_baseline",
        name="manual_baseline_modeld",
        reinit="finish_previous",
    ) as run:
        run.watch(model, log="gradients", log_freq=100)

        best_val_acc = 0.0
        best_epoch = -1

        for epoch in range(1, config["epochs"] + 1):
            train_loss, train_acc = train_one_epoch(
                model, train_loader, criterion, optimizer, DEVICE
            )
            val_loss, val_acc = validate_one_epoch(
                model, val_loader, criterion, DEVICE
            )

            history["train_loss"].append(train_loss)
            history["val_loss"].append(val_loss)
            history["train_acc"].append(train_acc)
            history["val_acc"].append(val_acc)

            if val_acc > best_val_acc:
                best_val_acc = val_acc
                best_epoch = epoch

            run.log({
                "epoch": epoch,
                "train_loss": train_loss,
                "val_loss": val_loss,
                "train_acc": train_acc,
                "val_acc": val_acc,
                "best_val_acc_so_far": best_val_acc,
            })

        run.summary["best_val_acc"] = best_val_acc
        run.summary["best_epoch"] = best_epoch

    return history, best_val_acc, best_epoch


# Run once if you want a fresh W&B-monitored manual baseline:
manual_history, manual_run_best_val_acc, manual_run_best_epoch = run_wandb_baseline(MANUAL_BASELINE_CONFIG)
print("Manual baseline best_val_acc:", manual_run_best_val_acc)
print("Manual baseline best_epoch  :", manual_run_best_epoch)

# =========================================================
# 8) Optuna objective with W&B logging per trial
# Compact search space, compact trial count
# =========================================================
HPO_EPOCHS = 12
N_TRIALS = 12

def objective(trial: optuna.trial.Trial) -> float:
    optimizer_name = trial.suggest_categorical("optimizer", ["adam", "rmsprop", "sgd_momentum"])
    batch_size = trial.suggest_categorical("batch_size", [32, 64])
    dropout = trial.suggest_categorical("dropout", [0.0, 0.2, 0.3])

    # Conditional lr range by optimizer
    if optimizer_name in ["adam", "rmsprop"]:
        lr = trial.suggest_float("lr", 1e-4, 3e-3, log=True)
    else:
        lr = trial.suggest_float("lr", 1e-3, 5e-2, log=True)

    weight_decay = trial.suggest_categorical("weight_decay", [0.0, 1e-5, 1e-4, 1e-3])

    train_loader = make_loader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = make_loader(val_dataset, batch_size=batch_size, shuffle=False)

    model = build_model_for_config(dropout=dropout).to(DEVICE)
    criterion = nn.CrossEntropyLoss()
    optimizer = build_optimizer(
        optimizer_name=optimizer_name,
        model=model,
        lr=lr,
        weight_decay=weight_decay,
    )

    best_val_acc = 0.0
    best_epoch = -1

    trial_config = {
        "trial_number": trial.number,
        "optimizer": optimizer_name,
        "batch_size": batch_size,
        "dropout": dropout,
        "lr": lr,
        "weight_decay": weight_decay,
        "epochs": HPO_EPOCHS,
    }

    with wandb.init(
        project=WANDB_PROJECT,
        entity=WANDB_ENTITY,
        config=trial_config,
        group="optuna_hpo",
        job_type="optuna_trial",
        name=f"optuna_trial_{trial.number:03d}",
        reinit="finish_previous",
    ) as run:
        run.watch(model, log="gradients", log_freq=100)

        for epoch in range(1, HPO_EPOCHS + 1):
            train_loss, train_acc = train_one_epoch(
                model, train_loader, criterion, optimizer, DEVICE
            )
            val_loss, val_acc = validate_one_epoch(
                model, val_loader, criterion, DEVICE
            )

            if val_acc > best_val_acc:
                best_val_acc = val_acc
                best_epoch = epoch

            run.log({
                "epoch": epoch,
                "train_loss": train_loss,
                "val_loss": val_loss,
                "train_acc": train_acc,
                "val_acc": val_acc,
                "best_val_acc_so_far": best_val_acc,
            })

            # Optuna pruning
            trial.report(best_val_acc, step=epoch)
            if trial.should_prune():
                run.summary["best_val_acc"] = best_val_acc
                run.summary["best_epoch"] = best_epoch
                run.summary["pruned"] = True
                raise TrialPruned()

        run.summary["best_val_acc"] = best_val_acc
        run.summary["best_epoch"] = best_epoch
        run.summary["pruned"] = False

    return best_val_acc

# =========================================================
# 9) Create and run the Optuna study
# direction='maximize' because we optimize validation accuracy
# =========================================================
sampler = optuna.samplers.TPESampler(seed=CFG.seed)
pruner = optuna.pruners.MedianPruner(n_startup_trials=4, n_warmup_steps=3)

study = optuna.create_study(
    study_name="modeld_hpo_wandb",
    storage=OPTUNA_STORAGE,
    load_if_exists=True,
    direction="maximize",
    sampler=sampler,
    pruner=pruner,
)

study.optimize(objective, n_trials=N_TRIALS)

print("Best trial number :", study.best_trial.number)
print("Best value        :", study.best_value)
print("Best params       :", study.best_trial.params)

# =========================================================
# 10) Trial results dataframe
# =========================================================
trials_df = study.trials_dataframe(attrs=("number", "value", "state", "params"))
trials_df = trials_df.sort_values(by="value", ascending=False).reset_index(drop=True)
trials_df.head(20)

# Prettier display
display(
    trials_df.style.format({
        "value": "{:.4f}",
    })
)

# =========================================================
# 11) Compare manual tuning vs Optuna
# =========================================================
comparison_df = pd.DataFrame([
    {
        "source": "manual_tuning",
        "best_val_acc": MANUAL_BEST_VAL_ACC,
        "notes": "Update this value if your manual best changes",
    },
    {
        "source": "manual_wandb_baseline_run",
        "best_val_acc": manual_run_best_val_acc,
        "notes": str(MANUAL_BASELINE_CONFIG),
    },
    {
        "source": "optuna_best_trial",
        "best_val_acc": study.best_value,
        "notes": str(study.best_trial.params),
    },
]).sort_values(by="best_val_acc", ascending=False).reset_index(drop=True)

comparison_df

# =========================================================
# 12) Optional final retrain of the best Optuna config
# Logged to W&B as the final selected run
# =========================================================
FINAL_EPOCHS = 25
best_params = study.best_trial.params

def run_final_best_optuna(best_params: dict):
    train_loader = make_loader(train_dataset, best_params["batch_size"], shuffle=True)
    val_loader = make_loader(val_dataset, best_params["batch_size"], shuffle=False)

    model = build_model_for_config(best_params["dropout"]).to(DEVICE)
    criterion = nn.CrossEntropyLoss()
    optimizer = build_optimizer(
        optimizer_name=best_params["optimizer"],
        model=model,
        lr=best_params["lr"],
        weight_decay=best_params["weight_decay"],
    )

    history = {
        "train_loss": [],
        "val_loss": [],
        "train_acc": [],
        "val_acc": [],
    }

    final_config = dict(best_params)
    final_config["epochs"] = FINAL_EPOCHS

    with wandb.init(
        project=WANDB_PROJECT,
        entity=WANDB_ENTITY,
        config=final_config,
        group="optuna_final",
        job_type="final_retrain",
        name="optuna_best_final_retrain",
        reinit="finish_previous",
    ) as run:
        run.watch(model, log="gradients", log_freq=100)

        best_val_acc = 0.0
        best_epoch = -1
        best_state = None

        for epoch in range(1, FINAL_EPOCHS + 1):
            train_loss, train_acc = train_one_epoch(
                model, train_loader, criterion, optimizer, DEVICE
            )
            val_loss, val_acc = validate_one_epoch(
                model, val_loader, criterion, DEVICE
            )

            history["train_loss"].append(train_loss)
            history["val_loss"].append(val_loss)
            history["train_acc"].append(train_acc)
            history["val_acc"].append(val_acc)

            if val_acc > best_val_acc:
                best_val_acc = val_acc
                best_epoch = epoch
                best_state = copy.deepcopy(model.state_dict())

            run.log({
                "epoch": epoch,
                "train_loss": train_loss,
                "val_loss": val_loss,
                "train_acc": train_acc,
                "val_acc": val_acc,
                "best_val_acc_so_far": best_val_acc,
            })

        run.summary["best_val_acc"] = best_val_acc
        run.summary["best_epoch"] = best_epoch

    if best_state is not None:
        model.load_state_dict(best_state)

    return model, history, best_val_acc, best_epoch


best_model, best_history, best_final_val_acc, best_final_epoch = run_final_best_optuna(best_params)
print("Final retrain best_val_acc:", best_final_val_acc)
print("Final retrain best_epoch  :", best_final_epoch)

# =========================================================
# 13) Simple plots for the final best Optuna run
# =========================================================
epochs = range(1, len(best_history["train_loss"]) + 1)

plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(epochs, best_history["train_loss"], label="train_loss")
plt.plot(epochs, best_history["val_loss"], label="val_loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Best Optuna config - Loss")
plt.grid(True)
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(epochs, best_history["train_acc"], label="train_acc")
plt.plot(epochs, best_history["val_acc"], label="val_acc")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Best Optuna config - Accuracy")
plt.grid(True)
plt.legend()

plt.tight_layout()
plt.show()

# =========================================================
# 14) Optional Optuna visual summaries
# Might require plotly in some environments
# =========================================================
try:
    import optuna.visualization as vis
    fig1 = vis.plot_optimization_history(study)
    fig2 = vis.plot_param_importances(study)
    fig1.show()
    fig2.show()
except Exception as e:
    print("Optuna visualization skipped:", e)

# =========================================================
# 15) Save results locally
# =========================================================
trials_csv = HPO_RESULTS_DIR / "optuna_trials_modeld.csv"
comparison_csv = HPO_RESULTS_DIR / "manual_vs_optuna_comparison.csv"

trials_df.to_csv(trials_csv, index=False)
comparison_df.to_csv(comparison_csv, index=False)

print("Saved trial results to :", trials_csv.resolve())
print("Saved comparison to    :", comparison_csv.resolve())



### Results and interpretation

The Optuna study identified **trial 11** as the best configuration within the compact **12-epoch screening phase**, reaching a validation accuracy of **0.5863**. The selected setup used **RMSprop**, **batch size 32**, **dropout 0.3**, **learning rate 0.000843**, and **weight decay 0.0**. This result already shows that the search was able to identify a meaningful and competitive region of the hyperparameter space, and that optimizer choice and regularization settings had a substantial effect on validation performance.

The trial table also reveals that the search space contained both promising and clearly unsuitable configurations. Several trials remained close to **0.10** validation accuracy, which is near chance level for a **10-class** classification problem. This indicates that some combinations of optimizer, learning rate, dropout, and weight decay were not viable. In that sense, the automated search was useful not only for finding a strong configuration, but also for identifying weak regions of the search space that would be poor candidates for further training.

The most important result is the **final retraining** of the best Optuna configuration. When the selected setup was trained for **25 epochs** instead of only **12**, it achieved a **best validation accuracy of 0.70117 at epoch 24**. This is clearly higher than the best short Optuna screening value and also higher than the earlier **manually tuned reference result of about 0.6668** from the previous tuning section. Under this experimental setup, the automatically selected configuration therefore outperformed the earlier manual reference.

It is important to distinguish between the **screening result** and the **final retrain result**. The Optuna trial value of **0.5863** was obtained under a deliberately limited budget of **12 epochs**, since the purpose of the study was efficient ranking of configurations rather than full convergence. By contrast, the final retrain used the same selected hyperparameters but a substantially larger training budget of **25 epochs**. The improvement from **0.5863** to **0.70117** is therefore expected and does not indicate inconsistency. Instead, it shows that Optuna successfully identified a promising configuration that performed much better once trained longer.

The final learning curves provide additional insight into the behaviour of the selected setup. Training loss decreases steadily and training accuracy rises to **0.86408**, while validation accuracy peaks at **0.70117** and ends at **0.68083**. At the same time, validation loss increases again towards the end and remains clearly above training loss. This suggests that the configuration optimizes effectively, but also begins to overfit during the later epochs. In other words, the automated search improved predictive performance, but it did not remove the generalization challenge entirely.

Compared with earlier manual tuning, the combined **Optuna + W&B** workflow offers three practical advantages. First, Optuna provides a more systematic and reproducible search strategy than manual trial-and-error. Second, W&B makes the behaviour of individual runs directly inspectable through logged training and validation curves. Third, the final retrain confirms that the automatically selected configuration was not only strong during the short screening phase, but also remained competitive under a larger training budget.

Overall, this section shows that combining **W&B for monitoring** with **Optuna for automated hyperparameter search** adds clear practical value. The automated workflow identified a configuration that surpassed the earlier manually tuned reference, while the logged curves made it possible to analyse convergence and overfitting behaviour in a structured and transparent way.

## 9. Optional Objective: Transfer Learning vs From-Scratch Training

This section investigates whether transfer learning can improve image classification performance compared with training models entirely from scratch. This addresses the optional objective of using **pre-trained models** and comparing them with models trained from scratch. 

To make the comparison meaningful, all variants in this section use the same **224x224 ImageNet-style preprocessing pipeline**. This is especially important because the pre-trained models were originally trained on ImageNet and therefore expect ImageNet-compatible normalization and resizing. Within this section, the input pipeline is therefore kept consistent across all compared variants.

The comparison includes both **from-scratch** and **pretrained** settings. In the pretrained case, two strategies are considered:
- **feature extraction**, where the pretrained backbone is frozen and only the final classification head is trained,
- **fine-tuning**, where the model is first trained with a frozen backbone and then further optimized with the full network unfrozen using a smaller learning rate.

This design allows the section to answer two related questions:  
first, whether pretrained representations provide an advantage over from-scratch training, and second, whether full fine-tuning improves upon simple feature extraction.

In [ ]:
# TRANSFER LEARNING VS FROM-SCRATCH COMPARISON
# Compare:
# - ModelD (custom baseline, from scratch)
# - ResNet18 (from scratch)
# - ResNet18 pretrained feature extraction
# - ResNet18 pretrained fine-tuning

# =========================================================

from __future__ import annotations

import copy
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, models
from torchvision.models import ResNet18_Weights, DenseNet121_Weights

# =========================================================
# 1) Paths + pretrained-compatible transforms
# Why this matters:
# pretrained ImageNet models expect ImageNet-style preprocessing.
# To keep this section comparable, we use the same 224-based
# transfer-learning transform for all variants in this experiment.
# =========================================================
TRAIN_ROOT = "data/train"
VAL_ROOT = "data/validate"

print("Train root:", TRAIN_ROOT)
print("Val root  :", VAL_ROOT)

# Official torchvision weights object provides the matching preprocessing pipeline
resnet_weights = ResNet18_Weights.DEFAULT
tl_transform = resnet_weights.transforms()

train_dataset_tl = datasets.ImageFolder(TRAIN_ROOT, transform=tl_transform)
val_dataset_tl   = datasets.ImageFolder(VAL_ROOT, transform=tl_transform)

print(f"Train samples: {len(train_dataset_tl):,}")
print(f"Val samples  : {len(val_dataset_tl):,}")
print(f"Classes      : {train_dataset_tl.classes}")

# =========================================================
# 2) DataLoader helper
# =========================================================

def make_loader(dataset, batch_size: int, shuffle: bool):
    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=CFG.num_workers,
        pin_memory=(DEVICE.type == "cuda"),
    )

# =========================================================
# 3) Shared comparison settings
# Keep these fixed for fair comparison
# =========================================================

TL_BATCH_SIZE = 32
SCRATCH_EPOCHS = 12
FEATURE_EXTRACT_EPOCHS = 12
FINETUNE_HEAD_EPOCHS = 4
FINETUNE_FULL_EPOCHS = 8
SCRATCH_LR = 1e-3
FEATURE_EXTRACT_LR = 1e-3
FINETUNE_LR = 1e-4
WEIGHT_DECAY = 0.0

INCLUDE_DENSENET = False  # set True if you also want DenseNet121 pretrained runs

train_loader_tl = make_loader(train_dataset_tl, batch_size=TL_BATCH_SIZE, shuffle=True)
val_loader_tl   = make_loader(val_dataset_tl,   batch_size=TL_BATCH_SIZE, shuffle=False)

print("Device:", DEVICE)

# =========================================================
# 4) Generic helpers
# =========================================================

def count_parameters(model: nn.Module):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable


def set_requires_grad(module: nn.Module, requires_grad: bool):
    for p in module.parameters():
        p.requires_grad = requires_grad
    
# =========================================================
# 5) Model builders
# =========================================================

def build_modeld_scratch():
    model = ModelD(
        in_channels=CFG.in_channels,
        num_classes=CFG.num_classes,
    )
    return model


def build_resnet18_scratch():
    model = models.resnet18(weights=None)
    in_features = model.fc.in_features
    model.fc = nn.Linear(in_features, CFG.num_classes)
    return model


def build_resnet18_pretrained_feature_extract():
    model = models.resnet18(weights=ResNet18_Weights.DEFAULT)
    set_requires_grad(model, False)
    in_features = model.fc.in_features
    model.fc = nn.Linear(in_features, CFG.num_classes)  # trainable by default
    return model


def build_resnet18_pretrained_finetune():
    model = models.resnet18(weights=ResNet18_Weights.DEFAULT)
    in_features = model.fc.in_features
    model.fc = nn.Linear(in_features, CFG.num_classes)
    set_requires_grad(model, True)
    return model


def build_densenet121_pretrained_feature_extract():
    model = models.densenet121(weights=DenseNet121_Weights.DEFAULT)
    set_requires_grad(model, False)
    in_features = model.classifier.in_features
    model.classifier = nn.Linear(in_features, CFG.num_classes)
    return model


def build_densenet121_pretrained_finetune():
    model = models.densenet121(weights=DenseNet121_Weights.DEFAULT)
    in_features = model.classifier.in_features
    model.classifier = nn.Linear(in_features, CFG.num_classes)
    set_requires_grad(model, True)
    return model
# =========================================================
# 6) Clean train / validate functions
# =========================================================

def train_one_epoch(
    model: nn.Module,
    loader,
    criterion: nn.Module,
    optimizer: torch.optim.Optimizer,
    device: torch.device,
):
    model.train()

    running_loss = 0.0
    running_correct = 0
    running_total = 0

    for images, targets in loader:
        images = images.to(device)
        targets = targets.to(device)

        optimizer.zero_grad()
        logits = model(images)
        loss = criterion(logits, targets)
        loss.backward()
        optimizer.step()

        batch_size = targets.size(0)
        running_loss += loss.item() * batch_size
        running_correct += (logits.argmax(dim=1) == targets).sum().item()
        running_total += batch_size

    return running_loss / running_total, running_correct / running_total


@torch.no_grad()
def validate_one_epoch(
    model: nn.Module,
    loader,
    criterion: nn.Module,
    device: torch.device,
):
    model.eval()

    running_loss = 0.0
    running_correct = 0
    running_total = 0

    for images, targets in loader:
        images = images.to(device)
        targets = targets.to(device)

        logits = model(images)
        loss = criterion(logits, targets)

        batch_size = targets.size(0)
        running_loss += loss.item() * batch_size
        running_correct += (logits.argmax(dim=1) == targets).sum().item()
        running_total += batch_size

    return running_loss / running_total, running_correct / running_total

# =========================================================
# 7) Generic fit function
# Trains only parameters with requires_grad=True
# =========================================================

def fit_variant(
    model: nn.Module,
    model_name: str,
    train_loader,
    val_loader,
    epochs: int,
    lr: float,
    weight_decay: float,
    device: torch.device,
    save_dir: Path,
):
    model = model.to(device)

    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(
        [p for p in model.parameters() if p.requires_grad],
        lr=lr,
        weight_decay=weight_decay,
    )

    history = {
        "model_name": model_name,
        "train_loss": [],
        "val_loss": [],
        "train_acc": [],
        "val_acc": [],
        "best_val_acc": 0.0,
        "best_val_loss": float("inf"),
        "best_epoch": -1,
    }

    total_params, trainable_params = count_parameters(model)
    history["num_parameters_total"] = total_params
    history["num_parameters_trainable"] = trainable_params

    best_state = None
    save_dir.mkdir(parents=True, exist_ok=True)

    print(f"\n=== Training {model_name} ===")
    print(f"Total params     : {total_params:,}")
    print(f"Trainable params : {trainable_params:,}")
    print(f"epochs={epochs}, lr={lr}, weight_decay={weight_decay}")

    for epoch in range(1, epochs + 1):
        train_loss, train_acc = train_one_epoch(
            model, train_loader, criterion, optimizer, device
        )
        val_loss, val_acc = validate_one_epoch(
            model, val_loader, criterion, device
        )

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["train_acc"].append(train_acc)
        history["val_acc"].append(val_acc)

        if val_acc > history["best_val_acc"]:
            history["best_val_acc"] = val_acc
            history["best_val_loss"] = val_loss
            history["best_epoch"] = epoch
            best_state = copy.deepcopy(model.state_dict())
            torch.save(best_state, save_dir / f"{model_name}_best.pt")

        print(
            f"Epoch {epoch:02d}/{epochs} | "
            f"train_loss={train_loss:.4f} | train_acc={train_acc:.4f} | "
            f"val_loss={val_loss:.4f} | val_acc={val_acc:.4f}"
        )

    if best_state is not None:
        model.load_state_dict(best_state)

    history["trained_model"] = model
    return history

# =========================================================
# 8) Two-stage fine-tuning helper
# Stage 1: train only head
# Stage 2: unfreeze full network and continue with lower lr
# =========================================================

def fit_two_stage_finetune_resnet18(
    model_name: str,
    train_loader,
    val_loader,
    head_epochs: int,
    full_epochs: int,
    head_lr: float,
    finetune_lr: float,
    weight_decay: float,
    device: torch.device,
    save_dir: Path,
):
    model = models.resnet18(weights=ResNet18_Weights.DEFAULT)
    set_requires_grad(model, False)
    in_features = model.fc.in_features
    model.fc = nn.Linear(in_features, CFG.num_classes)
    model = model.to(device)

    criterion = nn.CrossEntropyLoss()
    history = {
        "model_name": model_name,
        "train_loss": [],
        "val_loss": [],
        "train_acc": [],
        "val_acc": [],
        "best_val_acc": 0.0,
        "best_val_loss": float("inf"),
        "best_epoch": -1,
    }

    save_dir.mkdir(parents=True, exist_ok=True)
    best_state = None
    global_epoch = 0

    # ---- Stage 1: head only
    optimizer = torch.optim.Adam(
        [p for p in model.parameters() if p.requires_grad],
        lr=head_lr,
        weight_decay=weight_decay,
    )

    print(f"\n=== Training {model_name} | stage 1: head only ===")
    for _ in range(head_epochs):
        global_epoch += 1
        train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
        val_loss, val_acc = validate_one_epoch(model, val_loader, criterion, device)

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["train_acc"].append(train_acc)
        history["val_acc"].append(val_acc)

        if val_acc > history["best_val_acc"]:
            history["best_val_acc"] = val_acc
            history["best_val_loss"] = val_loss
            history["best_epoch"] = global_epoch
            best_state = copy.deepcopy(model.state_dict())

        print(
            f"Epoch {global_epoch:02d}/{head_epochs + full_epochs} | "
            f"train_loss={train_loss:.4f} | train_acc={train_acc:.4f} | "
            f"val_loss={val_loss:.4f} | val_acc={val_acc:.4f}"
        )

    # ---- Stage 2: full fine-tuning
    set_requires_grad(model, True)
    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=finetune_lr,
        weight_decay=weight_decay,
    )

    print(f"\n=== Training {model_name} | stage 2: full fine-tuning ===")
    for _ in range(full_epochs):
        global_epoch += 1
        train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
        val_loss, val_acc = validate_one_epoch(model, val_loader, criterion, device)

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["train_acc"].append(train_acc)
        history["val_acc"].append(val_acc)

        if val_acc > history["best_val_acc"]:
            history["best_val_acc"] = val_acc
            history["best_val_loss"] = val_loss
            history["best_epoch"] = global_epoch
            best_state = copy.deepcopy(model.state_dict())

        print(
            f"Epoch {global_epoch:02d}/{head_epochs + full_epochs} | "
            f"train_loss={train_loss:.4f} | train_acc={train_acc:.4f} | "
            f"val_loss={val_loss:.4f} | val_acc={val_acc:.4f}"
        )

    if best_state is not None:
        model.load_state_dict(best_state)

    total_params, trainable_params = count_parameters(model)
    history["num_parameters_total"] = total_params
    history["num_parameters_trainable"] = trainable_params
    history["trained_model"] = model

    torch.save(model.state_dict(), save_dir / f"{model_name}_best.pt")
    return history

# =========================================================
# 9) Experiment registry
# =========================================================

transfer_histories = {}
transfer_rows = []

SAVE_DIR_TRANSFER = Path(CFG.save_dir) / "transfer_learning_study"

transfer_experiments = [
    {
        "name": "ModelD_scratch",
        "runner": lambda: fit_variant(
            model=build_modeld_scratch(),
            model_name="ModelD_scratch",
            train_loader=train_loader_tl,
            val_loader=val_loader_tl,
            epochs=SCRATCH_EPOCHS,
            lr=SCRATCH_LR,
            weight_decay=WEIGHT_DECAY,
            device=DEVICE,
            save_dir=SAVE_DIR_TRANSFER,
        ),
    },
    {
        "name": "ResNet18_scratch",
        "runner": lambda: fit_variant(
            model=build_resnet18_scratch(),
            model_name="ResNet18_scratch",
            train_loader=train_loader_tl,
            val_loader=val_loader_tl,
            epochs=SCRATCH_EPOCHS,
            lr=SCRATCH_LR,
            weight_decay=WEIGHT_DECAY,
            device=DEVICE,
            save_dir=SAVE_DIR_TRANSFER,
        ),
    },
    {
        "name": "ResNet18_pretrained_feature_extract",
        "runner": lambda: fit_variant(
            model=build_resnet18_pretrained_feature_extract(),
            model_name="ResNet18_pretrained_feature_extract",
            train_loader=train_loader_tl,
            val_loader=val_loader_tl,
            epochs=FEATURE_EXTRACT_EPOCHS,
            lr=FEATURE_EXTRACT_LR,
            weight_decay=WEIGHT_DECAY,
            device=DEVICE,
            save_dir=SAVE_DIR_TRANSFER,
        ),
    },
    {
        "name": "ResNet18_pretrained_finetune",
        "runner": lambda: fit_two_stage_finetune_resnet18(
            model_name="ResNet18_pretrained_finetune",
            train_loader=train_loader_tl,
            val_loader=val_loader_tl,
            head_epochs=FINETUNE_HEAD_EPOCHS,
            full_epochs=FINETUNE_FULL_EPOCHS,
            head_lr=FEATURE_EXTRACT_LR,
            finetune_lr=FINETUNE_LR,
            weight_decay=WEIGHT_DECAY,
            device=DEVICE,
            save_dir=SAVE_DIR_TRANSFER,
        ),
    },
]

if INCLUDE_DENSENET:
    transfer_experiments.extend([
        {
            "name": "DenseNet121_pretrained_feature_extract",
            "runner": lambda: fit_variant(
                model=build_densenet121_pretrained_feature_extract(),
                model_name="DenseNet121_pretrained_feature_extract",
                train_loader=train_loader_tl,
                val_loader=val_loader_tl,
                epochs=FEATURE_EXTRACT_EPOCHS,
                lr=FEATURE_EXTRACT_LR,
                weight_decay=WEIGHT_DECAY,
                device=DEVICE,
                save_dir=SAVE_DIR_TRANSFER,
            ),
        },
        {
            "name": "DenseNet121_pretrained_finetune",
            "runner": lambda: fit_variant(
                model=build_densenet121_pretrained_finetune(),
                model_name="DenseNet121_pretrained_finetune",
                train_loader=train_loader_tl,
                val_loader=val_loader_tl,
                epochs=FEATURE_EXTRACT_EPOCHS,
                lr=FINETUNE_LR,
                weight_decay=WEIGHT_DECAY,
                device=DEVICE,
                save_dir=SAVE_DIR_TRANSFER,
            ),
        },
    ])

    
# =========================================================
# 10) Run all experiments
# =========================================================

for exp in transfer_experiments:
    print("=" * 90)
    print("Starting:", exp["name"])

    history = exp["runner"]()
    transfer_histories[exp["name"]] = history

    transfer_rows.append({
        "model_variant": exp["name"],
        "best_epoch": history["best_epoch"],
        "best_val_acc": history["best_val_acc"],
        "best_val_loss": history["best_val_loss"],
        "final_train_acc": history["train_acc"][-1],
        "final_val_acc": history["val_acc"][-1],
        "final_gap": history["train_acc"][-1] - history["val_acc"][-1],
        "num_parameters": history["num_parameters_total"],
        "trainable_parameters": history["num_parameters_trainable"],
        "epochs_trained": len(history["train_loss"]),
    })

transfer_df = pd.DataFrame(transfer_rows).sort_values(
    by="best_val_acc",
    ascending=False
).reset_index(drop=True)

# Prettier notebook display
display(
    transfer_df.style.format({
        "best_val_acc": "{:.4f}",
        "best_val_loss": "{:.4f}",
        "final_train_acc": "{:.4f}",
        "final_val_acc": "{:.4f}",
        "final_gap": "{:.4f}",
    })
)

# =========================================================
# 11) Plot helper: compare all variants together
# =========================================================

def plot_transfer_comparison(histories: dict) -> None:
    plt.figure(figsize=(12, 5))
    for name, hist in histories.items():
        epochs = range(1, len(hist["train_loss"]) + 1)
        plt.plot(epochs, hist["train_loss"], label=f"{name} train")
        plt.plot(epochs, hist["val_loss"], linestyle="--", label=f"{name} val")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title("Transfer learning comparison - Loss curves")
    plt.grid(True)
    plt.legend()
    plt.show()

    plt.figure(figsize=(12, 5))
    for name, hist in histories.items():
        epochs = range(1, len(hist["train_acc"]) + 1)
        plt.plot(epochs, hist["train_acc"], label=f"{name} train")
        plt.plot(epochs, hist["val_acc"], linestyle="--", label=f"{name} val")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.title("Transfer learning comparison - Accuracy curves")
    plt.grid(True)
    plt.legend()
    plt.show()

plot_transfer_comparison(transfer_histories)

# =========================================================
# 12) Single-run plots
# =========================================================

def plot_single_history(history: dict) -> None:
    model_name = history["model_name"]
    epochs = range(1, len(history["train_loss"]) + 1)

    plt.figure(figsize=(12, 4))

    plt.subplot(1, 2, 1)
    plt.plot(epochs, history["train_loss"], label="train_loss")
    plt.plot(epochs, history["val_loss"], label="val_loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title(f"{model_name} - Loss")
    plt.grid(True)
    plt.legend()

    plt.subplot(1, 2, 2)
    plt.plot(epochs, history["train_acc"], label="train_acc")
    plt.plot(epochs, history["val_acc"], label="val_acc")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.title(f"{model_name} - Accuracy")
    plt.grid(True)
    plt.legend()

    plt.tight_layout()
    plt.show()

for variant, history in transfer_histories.items():
    print("=" * 90)
    print(variant)
    plot_single_history(history)

# =========================================================
# 13) Best run summary
# =========================================================

best_transfer_run = transfer_df.iloc[0]

print("Best transfer-learning comparison run")
print("-" * 40)
print(f"Model variant        : {best_transfer_run['model_variant']}")
print(f"Best epoch           : {int(best_transfer_run['best_epoch'])}")
print(f"Best val acc         : {best_transfer_run['best_val_acc']:.4f}")
print(f"Best val loss        : {best_transfer_run['best_val_loss']:.4f}")
print(f"Final train acc      : {best_transfer_run['final_train_acc']:.4f}")
print(f"Final val acc        : {best_transfer_run['final_val_acc']:.4f}")
print(f"Final gap            : {best_transfer_run['final_gap']:.4f}")
print(f"Total params         : {int(best_transfer_run['num_parameters']):,}")
print(f"Trainable params     : {int(best_transfer_run['trainable_parameters']):,}")
print(f"Epochs trained       : {int(best_transfer_run['epochs_trained'])}")

# =========================================================
# 14) Save results
# =========================================================

results_dir = Path("./results")
results_dir.mkdir(parents=True, exist_ok=True)

csv_path = results_dir / "transfer_learning_comparison.csv"
transfer_df.to_csv(csv_path, index=False)

print("Saved transfer learning comparison results to:", csv_path.resolve())

### Results and interpretation

The transfer-learning results show a very strong benefit from using pretrained ImageNet representations. Both pretrained ResNet18 variants achieved exceptionally high validation performance within a compact training budget of **12 epochs**, clearly demonstrating the practical value of transfer learning on this task.

The strongest result was obtained with **pretrained fine-tuning**. `ResNet18_pretrained_finetune` reached a best validation accuracy of **0.9468** at **epoch 7**, with a best validation loss of **0.1988**. This was the highest peak performance among all variants in this section. The result suggests that unfreezing the full network after the initial head training allowed the pretrained representation to adapt more closely to the iCoSimal task. At the same time, the final train-validation gap (**0.0467**) and the very high final training accuracy (**0.9885**) indicate that the model became more prone to later overfitting.

The **pretrained feature-extraction** setup was nearly as strong while being much more parameter-efficient. `ResNet18_pretrained_feature_extract` achieved a best validation accuracy of **0.9447** at **epoch 12**, with the lowest best validation loss in this comparison (**0.1730**). Although the full model contains **11,181,642** parameters, only **5,130** were trainable. This is a highly efficient result: only the classifier head was optimized, yet the model remained extremely strong and stable. The final train-validation gap was essentially zero (**-0.0022**), which further supports the interpretation that feature extraction was the most stable and efficient pretrained strategy.

The contrast to the **from-scratch** runs is particularly important. `ResNet18_scratch` already performed substantially better than `ModelD_scratch`, reaching a best validation accuracy of **0.7768** compared with **0.6322** for `ModelD_scratch`. This shows that architecture alone does matter: even without pretraining, the deeper ResNet18 backbone was clearly stronger than the simpler custom CNN under the same preprocessing pipeline and training budget. However, `ResNet18_scratch` still remained far below both pretrained ResNet18 variants. In addition, it showed a much larger final train-validation gap (**0.2191**) than the pretrained models, despite reaching a very high final training accuracy (**0.9386**). This indicates that the main gain in this section comes not only from architecture, but from the transferred pretrained representation.

`ModelD_scratch` serves as the weakest baseline in this comparison. It achieved a best validation accuracy of **0.6322** at **epoch 12**, with a best validation loss of **1.0639**. Its final train and validation accuracies remained close (**0.6439** vs. **0.6322**), resulting in a very small final gap (**0.0118**). This suggests that the model was comparatively stable, but also clearly limited in performance under the current training budget.

These results therefore support three distinct conclusions:
- **highest peak performance:** achieved by **pretrained fine-tuning**,
- **strongest stability and trainable-parameter efficiency:** achieved by **pretrained feature extraction**,
- **strongest from-scratch model:** achieved by **ResNet18_scratch**, which was clearly better than `ModelD_scratch` but still far below the pretrained variants.

From a deep-learning perspective, this pattern is highly plausible. ImageNet pretraining provides a rich prior of transferable visual features such as edges, textures, shapes, and object parts. Instead of learning these representations from scratch, the pretrained models can start from a strong visual basis and adapt it to the animal-classification task. This is especially advantageous when the training budget is limited, because the model does not need to rediscover useful low- and mid-level features on its own.

Overall, this experiment provides clear evidence that **transfer learning is highly effective** for the iCoSimal V3 classification problem. While a stronger architecture already helps in the from-scratch setting, the dominant improvement comes from pretrained representations. Pretrained feature extraction already gives excellent and highly efficient performance, while pretrained fine-tuning delivers the strongest peak result. Under limited training time, transfer learning is therefore the most practical and most powerful strategy in this comparison.

## 10. Conclusion

This work investigated convolutional neural networks on the iCoSimal V3 dataset through a sequence of controlled experiments covering both mandatory and optional objectives. The overall goal was not only to obtain strong classification performance, but also to understand how architecture design, hyperparameter choices, regularization, optimization, and transfer learning influence model behaviour.

Across the full study, several consistent insights emerged. Model performance was strongly affected by the interaction between architectural capacity and training setup. Better architectures alone were not always sufficient, and seemingly weak results could often be improved substantially through better optimization or more suitable hyperparameters. At the same time, stronger models also increased the risk of overfitting, which made careful regularization and monitoring important.

### Main takeaways from each part

- **Fair depth comparison**  
  Increasing depth and capacity improved performance under the same initial training setup. The comparison showed that deeper CNNs can learn more useful representations, but also require a suitable training budget before their advantage becomes visible.

- **Hyperparameter tuning**  
  The same architecture performed very differently depending on the selected learning rate, batch size, and weight decay. This showed that model quality depends not only on architecture, but also heavily on the training configuration.

- **Underfitting, good fit, and overfitting**  
  These experiments made the generalization regimes very clear. A model with insufficient capacity underfit the task, a stronger model on the full dataset provided a good fit, and the same strong model trained on only a small subset showed clear overfitting behaviour.

- **Regularization experiments**  
  Regularization was useful, but not automatically beneficial. Different methods improved different aspects of training. Dropout improved peak validation accuracy, mild weight decay gave the most balanced overall generalization behaviour, and early stopping helped limit late overfitting. Stronger regularization could become too restrictive.

- **Optimizer comparison**  
  The optimizer had a major influence on convergence speed and final performance. Adaptive optimizers clearly outperformed the SGD variants under the chosen setup, which showed that optimization should be treated as an important modelling decision rather than a minor implementation detail.

- **Advanced architectures**  
  More sophisticated architectures did not automatically dominate a simpler custom model. DenseNet121 achieved the strongest peak result, but ModelD remained competitive while being much smaller and more stable. This highlighted the trade-off between raw capacity, stability, and efficiency.

- **W&B and Optuna**  
  Monitoring and automated search added clear practical value. W&B made it possible to inspect and compare training dynamics systematically, while Optuna identified a configuration that outperformed earlier manual tuning. This showed that structured experiment tracking and automated optimization are highly useful in real deep-learning workflows.

- **Transfer learning**  
  Transfer learning produced the strongest overall results. Pretrained feature extraction already performed extremely well with very few trainable parameters, while fine-tuning achieved the highest peak performance. This demonstrated the practical strength of pretrained representations for vision tasks.

### Overall conclusion

The experiments show that strong deep-learning performance is not the result of a single design choice. Instead, it emerges from the combination of a suitable architecture, a well-tuned optimization setup, controlled regularization, and careful monitoring of training behaviour. Among all investigated directions, transfer learning provided the strongest performance gains, while systematic tuning and experiment tracking greatly improved the reliability and interpretability of the workflow. Overall, this work demonstrates that successful CNN modelling requires both technical experimentation and careful analytical interpretation. 